# 🎼 Quantum Music Composer – interactive dashboard
Compose a harmonious piece (**at least 20 seconds**) with quantum circuits – and change key, scale, tempo, number of notes, rhythm, time signature, genre-specific arrangements, swing, guitar parts and drums from a dashboard instead of editing code.

**How to use:** `Runtime → Run all` (or run the cells top to bottom). The dashboard appears under the cell titled *Dashboard*. Pick your settings, press **🎼 Compose**, listen, and download the WAV / MIDI.

| Musical job | Quantum algorithm | Reference |
|---|---|---|
| Chord progression | **Hadamard QRNG** (superposition + measurement) | Yang & Lee §2.2, 3.3 |
| **Rhythm** of every bar | **Partitioned Quantum Cellular Automaton (PQCA)** | Miranda §6.2 |
| **Melody** pitch-by-pitch | **QuPoly-style quantum Markov chain** (rules → amplitudes → `StatePreparation` → measurement) | Miranda §6.1 |
| Harmonic strong-beat notes | **Grover / amplitude amplification** boosting chord tones | Miranda §6.1, Yang & Lee §2.1 |
| Accompaniment decisions | **Bell-pair entanglement** (H + CNOT) | Yang & Lee §3.3 |
| Sound | FM synthesis (as in Q1Synth) → WAV + MIDI | Miranda §5 |

**Why it sounds good:** the quantum computer chooses *which* musical option; classical music theory (scale, chords built from the scale, chord-tone targeting, stepwise motion, leap recovery) guarantees *what options exist*.

## 1. Install

In [ ]:
!pip install -q qiskit qiskit-aer scipy matplotlib pylatexenc mido ipywidgets

## 2. Imports & simulator

In [ ]:
import math, io, base64, warnings, traceback
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.signal import fftconvolve, butter, sosfilt
from scipy.io import wavfile
from IPython.display import Audio, display, clear_output, HTML
import ipywidgets as W

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
try:                                   # Qiskit >= 2.1
    from qiskit.circuit.library import DiagonalGate as _Diag
    def Diag(d): return _Diag(d)
except ImportError:                    # older Qiskit
    from qiskit.circuit.library import Diagonal as _Diag
    def Diag(d): return _Diag(d).to_gate()
from qiskit_aer import AerSimulator
from mido import Message, MetaMessage, MidiFile, MidiTrack, bpm2tempo

warnings.filterwarnings("ignore")

SR  = 44100                      # audio sample rate
SIM = AerSimulator()
CFG = {"seed": 7}               # None -> a different piece every run
_run_id = [0]

def run_circuit(qc, shots=1):
    """Run a circuit on the quantum simulator and return the measured bitstrings (one per shot)."""
    _run_id[0] += 1
    t = transpile(qc, basis_gates=["u", "cx"], optimization_level=1)   # full decomposition -> robust on every Aer version
    kw = {} if CFG["seed"] is None else {"seed_simulator": CFG["seed"] * 100003 + _run_id[0]}
    return SIM.run(t, shots=shots, memory=True, **kw).result().get_memory()

print("Ready: libraries loaded.")

Ready: libraries loaded.


## 3. Musical frame – now data-driven
Nothing is hard-coded to A minor any more. A **key** + a **scale** generate everything:
* the melody lexicon (n consecutive scale notes around C5),
* seven diatonic chords (thirds stacked on each scale degree; optional 7ths),
* four chord loops per scale that avoid diminished chords and end on the tonic,
* time signatures (eighth-note slots, strong beats, bass patterns) and style presets (tempo, scale, density, sound).

In [ ]:
NOTE_NAMES = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
def name(m): return f"{NOTE_NAMES[m % 12]}{m // 12 - 1}"

# ---------- scales: intervals above the tonic + the chord loops (as scale-degree numbers, 0 = tonic) ----------
# 'parent' = 7-note scale used to build the chords; 'melody' = notes the melody may use (pentatonics drop two notes)
_MAJ  = [0, 2, 4, 5, 7, 9, 11];  _MIN = [0, 2, 3, 5, 7, 8, 10]
PROG_MAJOR = [[0,4,5,3], [0,5,3,4], [0,3,4,3], [0,3,5,4]]          # I-V-vi-IV, I-vi-IV-V, I-IV-V-IV, I-IV-vi-V
PROG_MINOR = [[0,5,2,6], [0,3,6,2], [0,6,5,6], [0,2,6,5]]          # i-VI-III-VII, i-iv-VII-III, i-VII-VI-VII, i-III-VII-VI
SCALES = {
  "Major (bright)":               dict(parent=_MAJ, melody=_MAJ, progs=PROG_MAJOR),
  "Natural minor (sad)":          dict(parent=_MIN, melody=_MIN, progs=PROG_MINOR),
  "Dorian (jazzy minor)":         dict(parent=[0,2,3,5,7,9,10], melody=[0,2,3,5,7,9,10], progs=[[0,3,0,6],[0,3,6,3],[0,6,3,0],[0,1,3,0]]),
  "Phrygian (dark)":              dict(parent=[0,1,3,5,7,8,10], melody=[0,1,3,5,7,8,10], progs=[[0,1,0,6],[0,1,2,6],[0,5,6,1],[0,3,1,0]]),
  "Lydian (dreamy)":              dict(parent=[0,2,4,6,7,9,11], melody=[0,2,4,6,7,9,11], progs=[[0,1,0,4],[0,4,5,1],[0,1,4,0],[0,5,1,4]]),
  "Mixolydian (bluesy major)":    dict(parent=[0,2,4,5,7,9,10], melody=[0,2,4,5,7,9,10], progs=[[0,6,3,0],[0,3,6,3],[0,4,3,6],[0,1,3,6]]),
  "Major pentatonic (folk)":      dict(parent=_MAJ, melody=[0,2,4,7,9],  progs=PROG_MAJOR),
  "Minor pentatonic (safe, moody)": dict(parent=_MIN, melody=[0,3,5,7,10], progs=PROG_MINOR),
}

def build_chords(tonic, parent, sevenths=False):
    """Stack thirds on every degree of the parent scale -> {degree: label, root, tones (pitch classes), dim?}"""
    pcs = [(tonic + i) % 12 for i in parent]
    out = {}
    for d in range(7):
        idx = [d, (d + 2) % 7, (d + 4) % 7] + ([(d + 6) % 7] if sevenths else [])
        tones = [pcs[j] for j in idx]
        r = tones[0]; third = (tones[1] - r) % 12; fifth = (tones[2] - r) % 12
        qual = "" if third == 4 else ("m" if fifth == 7 else "dim")
        ext = ""
        if sevenths:
            s = (tones[3] - r) % 12
            if   qual == ""  and s == 11: ext = "maj7"
            elif qual == "m" and s == 11: ext = "(maj7)"
            elif qual == "dim" and s == 10: ext = "ø7"
            else: ext = "7"
        out[d] = dict(label=NOTE_NAMES[r] + qual + ext, root=r, tones=tones, dim=(qual == "dim"))
    return out

def build_lexicon(tonic, mel_intervals, n):
    """n consecutive scale notes centred near C5 that contain the tonic (so the piece can end on it)."""
    pcs = {(tonic + i) % 12 for i in mel_intervals}
    cands = [m for m in range(48, 97) if m % 12 in pcs]
    best, best_score = None, 1e9
    for s in range(len(cands) - n + 1):
        win = cands[s:s + n]
        if not any(m % 12 == tonic for m in win): continue
        score = abs(win[n // 2] - 72)
        if score < best_score: best, best_score = win, score
    return best

# ---------- time signatures (one slot = one eighth note) ----------
TIMESIGS = {
  "4/4": dict(slots=8, strong=(0, 4), num=4, den=4, plain=[(0,4,"r"),(4,4,"f")],            sync=[(0,3,"r"),(3,3,"r"),(6,2,"f")]),
  "3/4": dict(slots=6, strong=(0,),   num=3, den=4, plain=[(0,2,"r"),(2,4,"f")],            sync=[(0,3,"r"),(3,3,"r")]),
  "6/8": dict(slots=6, strong=(0, 3), num=6, den=8, plain=[(0,3,"r"),(3,3,"f")],            sync=[(0,2,"r"),(2,2,"r"),(4,2,"f")]),
  "2/4": dict(slots=4, strong=(0,),   num=2, den=4, plain=[(0,2,"r"),(2,2,"f")],            sync=[(0,1,"r"),(1,1,"r"),(2,2,"f")]),
}

# ---------- rhythmic busyness -> how many of the cells may be 'on' in a bar (fractions of the bar) ----------
BUSY = {"Sparse": (0.15, 0.40), "Medium": (0.375, 0.75), "Busy": (0.60, 0.90)}

# ---------- sound design and genre presets ----------
# These are synthesis envelopes / arrangement choices, not claims of real instrument sampling.
SOUNDS = {
  "classic": dict(idx0=2.5, idx_decay=3.0, idx_base=0.4, mel_attack=0.004, mel_release=0.30, mel_curve=1.1,
                  reverb=0.18, rt=2.8, arp=True, arp_vel=38, pad_vel=42, pad_gain=0.35, bass="pattern", lowpass=None,
                  lead="synth", pad="soft", drive=1.0, vinyl=0.0, wow=0.0),
  "lofi":    dict(idx0=0.9, idx_decay=1.8, idx_base=0.15, mel_attack=0.012, mel_release=0.34, mel_curve=1.1,
                  reverb=0.12, rt=1.8, arp=False, arp_vel=24, pad_vel=34, pad_gain=0.22, bass="pattern", lowpass=2200,
                  lead="epiano", pad="warm", drive=1.35, vinyl=0.025, wow=0.0025),
  "ambient": dict(idx0=0.7, idx_decay=0.8, idx_base=0.2, mel_attack=0.22, mel_release=2.0, mel_curve=0.18,
                  reverb=0.62, rt=5.0, arp=False, arp_vel=22, pad_vel=72, pad_gain=0.72, bass="sustain", lowpass=6500,
                  lead="soft_sine", pad="evolving", drive=1.0, vinyl=0.0, wow=0.0),
  "edm":     dict(idx0=4.0, idx_decay=5.0, idx_base=0.8, mel_attack=0.002, mel_release=0.12, mel_curve=2.2,
                  reverb=0.10, rt=2.2, arp=True, arp_vel=55, pad_vel=42, pad_gain=0.28, bass="eighths", lowpass=None,
                  lead="bright_synth", pad="sidechain", drive=1.1, vinyl=0.0, wow=0.0),
  "folk":    dict(idx0=1.5, idx_decay=6.0, idx_base=0.05, mel_attack=0.002, mel_release=0.20, mel_curve=3.0,
                  reverb=0.10, rt=2.0, arp=False, arp_vel=30, pad_vel=28, pad_gain=0.16, bass="alternating", lowpass=None,
                  lead="fiddle", pad="none", drive=1.0, vinyl=0.0, wow=0.0),
  "jazz":    dict(idx0=1.8, idx_decay=2.8, idx_base=0.25, mel_attack=0.025, mel_release=0.22, mel_curve=0.65,
                  reverb=0.12, rt=2.0, arp=False, arp_vel=24, pad_vel=40, pad_gain=0.18, bass="walking", lowpass=None,
                  lead="sax", pad="none", drive=1.0, vinyl=0.0, wow=0.0),
  "rock":    dict(idx0=4.8, idx_decay=3.8, idx_base=1.0, mel_attack=0.003, mel_release=0.18, mel_curve=1.8,
                  reverb=0.10, rt=2.2, arp=False, arp_vel=32, pad_vel=24, pad_gain=0.12, bass="pattern", lowpass=None,
                  lead="rock_guitar", pad="none", drive=1.8, vinyl=0.0, wow=0.0),
}

# Genre defaults are starting points, not hard definitions: users can override the controls.
STYLES = {
  "Pop / cinematic (default)": dict(bpm=108, scale="Major (bright)", busy="Medium", notes=10, ts="4/4", sevenths=False, sound="classic", swing=0.50, drums="pop"),
  "EDM / folk-house":          dict(bpm=124, scale="Major (bright)", busy="Busy", notes=9, ts="4/4", sevenths=False, sound="edm", swing=0.50, drums="edm"),
  "Country":                   dict(bpm=104, scale="Major pentatonic (folk)", busy="Medium", notes=8, ts="4/4", sevenths=False, sound="folk", swing=0.50, drums="country"),
  "Jazz (swing)":              dict(bpm=118, scale="Dorian (jazzy minor)", busy="Medium", notes=12, ts="4/4", sevenths=True, sound="jazz", swing=0.667, drums="jazz"),
  "Lo-fi hip-hop":             dict(bpm=78, scale="Dorian (jazzy minor)", busy="Sparse", notes=8, ts="4/4", sevenths=True, sound="lofi", swing=0.58, drums="lofi"),
  "Ambient":                   dict(bpm=66, scale="Lydian (dreamy)", busy="Sparse", notes=9, ts="4/4", sevenths=True, sound="ambient", swing=0.50, drums="none"),
  "Rock":                      dict(bpm=132, scale="Minor pentatonic (safe, moody)", busy="Busy", notes=10, ts="4/4", sevenths=False, sound="rock", swing=0.50, drums="rock"),
}

# quick self-check: print the chords for A natural minor
_ch = build_chords(9, _MIN)
print("A natural minor chords:", {d: c["label"] for d, c in _ch.items()})
print("Melody lexicon (11 notes):", [name(m) for m in build_lexicon(9, _MIN, 11)])

A natural minor chords: {0: 'Am', 1: 'Bdim', 2: 'C', 3: 'Dm', 4: 'Em', 5: 'F', 6: 'G'}
Melody lexicon (11 notes): ['E4', 'F4', 'G4', 'A4', 'B4', 'C5', 'D5', 'E5', 'F5', 'G5', 'A5']


## 4. Quantum building blocks
The circuits are the same for every setting – only their size changes (e.g. the Markov circuit uses ⌈log₂(#pitches)⌉ qubits, the PQCA uses one qubit per eighth-note slot).

**Grover on strong beats:** with `A` the state-preparation unitary and χ the chord tones, `Q = A·S₀·A†·S_χ`. After `k ≈ π/(4θ) − ½` iterations the chord-tone probability is `sin²((2k+1)θ)`. We pre-condition the chord-tone mass to 0.30 so that one iteration gives ≈ 97 %, while relative weights inside each group stay untouched.

In [ ]:
# ---------------- 1. Hadamard QRNG ----------------
def qrng_circuit(n):
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

def qrng_bits(n):
    return run_circuit(qrng_circuit(n))[0]

# ---------------- 2. PQCA rhythm ----------------
def pqca_circuit(state):
    n = len(state)
    qc = QuantumCircuit(n, n)
    for i, b in enumerate(state):          # arm qubits with last measurement
        if b: qc.x(i)
    qc.barrier()
    for i in range(0, n, 2):               # frame 1: CRX(pi/4) inside supercells
        qc.crx(np.pi/4, i, i+1)
    qc.barrier()
    for i in range(1, n, 2):               # frame 2: shifted partition (wraps around)
        j = (i + 1) % n
        qc.h(i); qc.cx(i, j)
    qc.barrier()
    qc.measure(range(n), range(n))
    return qc

def pqca_step(state):
    bits = run_circuit(pqca_circuit(state))[0]       # string is q_{n-1}...q_0
    return [int(b) for b in bits[::-1]]

def quantum_rhythm(bars, slots, min_on, max_on):
    seed_state = [1 if i in (0, 3, 5) else 0 for i in range(slots)]
    state, rhythm = seed_state, []
    for _ in range(bars):
        for _try in range(20):                       # re-measure if the bar is too sparse/busy
            nxt = pqca_step(state)
            if min_on <= sum(nxt) <= max_on: break
        state = nxt
        pattern = nxt.copy(); pattern[0] = 1         # a note always starts the bar
        rhythm.append(pattern)
    return rhythm

# ---------------- 3. Quantum Markov chain (QuPoly) + Grover amplitude amplification ----------------
def transition_weights(mel, nq, prev_i, prev_dir, tones, strong, genre="Pop / cinematic (default)"):
    """Classical genre-aware musical rules -> amplitudes/probabilities for quantum sampling."""
    n = len(mel); w = np.zeros(2**nq)
    ctones = {i for i, m in enumerate(mel) if m % 12 in tones}
    centre = (n - 1) / 2; sigma = max(1.5, 3.0 * n / 11)
    for i, m in enumerate(mel):
        d = abs(i - prev_i)
        x = {0: 0.2, 1: 3.0, 2: 2.0, 3: 0.8, 4: 0.45}.get(d, 0.12 if d == 5 else 0.04)
        # Genre-specific transition preferences. These shape the distribution;
        # the subsequent state preparation and measurement remain quantum-circuit operations.
        if genre == "Jazz (swing)":
            if d in (2, 3, 4): x *= 1.35
            if not strong and i in ctones: x *= 1.7
        elif genre == "Rock":
            if d >= 3: x *= 1.35
            if d == 0: x *= 0.75
        elif genre == "Lo-fi chill":
            if d <= 2: x *= 1.35
            if d >= 4: x *= 0.55
        elif genre == "Ambient":
            if d <= 2: x *= 1.5
            if d >= 4: x *= 0.45
        elif genre == "EDM / dance":
            if d in (0, 2): x *= 1.25
        else:  # Pop and Folk: memorable motifs with mostly stepwise movement
            if d == 0: x *= 1.2
            if d >= 4: x *= 0.75
        if not strong and i in ctones: x *= 1.5
        # Strong-beat chord-tone preference is primarily supplied by Grover.
        if not strong and i not in ctones and any((m - c) % 12 in (1, 11) for c in tones):
            x *= 0.5
        x *= math.exp(-((i - centre) ** 2) / (2 * sigma ** 2))
        if abs(prev_dir) >= 3 and d in (1, 2) and np.sign(i - prev_i) == -np.sign(prev_dir):
            x *= 2.5
        w[i] = x
    return w / w.sum()

def strong_prior(probs, marked, p0=0.30):
    """Pre-condition so the chord-tone mass is p0 (Grover works best for small p). Relative weights inside each group are untouched."""
    mask = np.zeros(len(probs), bool); mask[list(marked)] = True
    a, b = probs[mask].sum(), probs[~mask].sum()
    if a <= 0 or b <= 0: return None
    pr = probs.copy(); pr[mask] *= p0 / a; pr[~mask] *= (1 - p0) / b
    return pr

def amplitude_amplification_circuit(probs, marked, k=None):
    """A, S_chi, S_0 as in Grover; k chosen optimally if not given."""
    N = len(probs); n = int(np.log2(N))
    p = float(sum(probs[i] for i in marked))
    if k is None:
        k = 0
        if 0 < p < 1:
            theta = math.asin(math.sqrt(p))
            k = max(0, int(round(math.pi / (4 * theta) - 0.5)))
    A = StatePreparation(np.sqrt(probs))
    s_chi = Diag([-1 if i in marked else 1 for i in range(N)])
    s_0   = Diag([1] + [-1] * (N - 1))                  # reflection about |0..0> (up to global phase)
    qc = QuantumCircuit(n, n)
    qc.append(A, range(n))
    for _ in range(k):
        qc.append(s_chi, range(n))
        qc.append(A.inverse(), range(n))
        qc.append(s_0, range(n))
        qc.append(A, range(n))
    qc.measure(range(n), range(n))
    return qc, k, p

def p_success(p, k):                                    # theory: sin^2((2k+1) theta)
    return math.sin((2 * k + 1) * math.asin(math.sqrt(p))) ** 2

def quantum_note(mel, nq, prev_i, prev_dir, tones, strong, force=None, genre="Pop / cinematic (default)"):
    """One melody note = one quantum circuit. Returns (index, circuit, k, p_before, distribution_used)."""
    n = len(mel)
    probs = transition_weights(mel, nq, prev_i, prev_dir, tones, strong, genre)
    marked = [i for i, m in enumerate(mel) if m % 12 in tones]
    if force is not None:                               # final cadence: only the tonic
        probs = np.zeros(2**nq); probs[force] = 1.0
        qc, k, p = amplitude_amplification_circuit(probs, marked, k=0)
    else:
        pr = strong_prior(probs, marked) if (strong and 0 < len(marked) < n) else None
        if pr is not None:
            probs = pr
            qc, k, p = amplitude_amplification_circuit(probs, marked)           # strong beat: Grover
        else:
            qc, k, p = amplitude_amplification_circuit(probs, marked, k=0)      # weak beat: plain QuPoly sampling
    idx = int(run_circuit(qc)[0], 2)
    if idx >= n: idx = int(np.argmax(probs))             # safety net (cannot normally happen)
    return idx, qc, k, p, probs

# ---------------- 4. Bell-pair entanglement ----------------
def bell_circuit():
    qc = QuantumCircuit(2, 2)
    qc.h(0); qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

def bell_pairs(bars):
    res = run_circuit(bell_circuit(), shots=bars)       # one independent outcome per bar
    return [(int(s[1]), int(s[0])) for s in res]        # (q0, q1) - always equal

print("Quantum building blocks defined.")

Quantum building blocks defined.


## 5. Composer – settings in, piece out

In [ ]:
def make_drum_pattern(genre, bars, slots):
    """Genre-specific GM percussion: (absolute_slot, duration, GM_pitch, velocity)."""
    if genre == "none": return []
    # Templates are defined on an 8-eighth-note 4/4 bar.
    templates = {
      "jazz": [(i,51,58 if i in (0,4) else 42) for i in range(8)] + [(2,37,28),(6,37,32)],
      "rock": [(0,36,112),(4,36,100),(2,38,108),(6,38,112)] + [(i,42,48) for i in range(8)],
      "pop":  [(0,36,92),(4,36,82),(2,38,92),(6,38,92)] + [(i,42,38) for i in (0,2,4,6)],
      "edm":  [(i,36,118) for i in (0,2,4,6)] + [(2,39,86),(6,39,90)] + [(i,42,52) for i in range(8)],
      "lofi": [(0,36,62),(4,36,48),(2,37,46),(6,37,48)] + [(i,42,24 if i%2==0 else 15) for i in range(8)],
      "country": [(0,36,88),(4,36,72),(2,38,70),(6,38,78)] + [(i,42,36 if i%2==0 else 25) for i in range(8)],
    }
    template = templates.get(genre, [])
    events=[]
    for bar in range(bars):
        # EDM arrangement: pull back in the intro, build in the second section,
        # then restore full-energy kick/clap/hat. This is an original arrangement.
        section = bar / max(1, bars)
        for slot,pitch,vel in template:
            if genre == "edm":
                if section < 0.20 and pitch == 36 and slot != 0: continue
                if 0.55 <= section < 0.70: vel = int(vel * 0.65)
                if section >= 0.70: vel = min(127, int(vel * 1.05))
            if genre == "country" and pitch == 42 and slot % 2 == 1:
                vel = int(vel * 0.85)
            mapped = int(round(slot * slots / 8)) % slots
            events.append((bar*slots+mapped,1,pitch,int(vel)))
    return events


def compose(p, progress=lambda frac, msg="": None):
    """Build a hybrid quantum/classical composition from dashboard settings."""
    CFG["seed"] = None if p["random_seed"] else int(p["seed"])
    _run_id[0] = 0
    genre = p.get("genre", "Pop / cinematic (default)")
    style = dict(STYLES[genre])
    style["swing"] = float(p.get("swing", style["swing"]))
    sc, ts = SCALES[p["scale"]], TIMESIGS[p["ts"]]
    tonic, slots = p["key"], ts["slots"]
    slot_sec = 30.0 / p["bpm"]
    bar_sec = slots * slot_sec
    secs = max(20, p["seconds"])
    bars = max(4, int(math.ceil(math.ceil(secs / bar_sec) / 4) * 4))
    total = bars * bar_sec
    S = SOUNDS[p["sound"]]
    CH = build_chords(tonic, sc["parent"], p["sevenths"])
    MEL = build_lexicon(tonic, sc["melody"], p["notes"])
    NQ = max(2, int(math.ceil(math.log2(len(MEL)))))
    final_i = min((i for i, m in enumerate(MEL) if m % 12 == tonic), key=lambda i: abs(i - len(MEL) // 2))

    progress(0.02, "QRNG picks the chord progression ...")
    bits = qrng_bits(2); prog_id = int(bits, 2)
    degs = sc["progs"][prog_id] * (bars // 4)
    # Genre-specific harmonic loops (scale degrees). QRNG still selects a loop;
    # these are different valid choices for each arrangement family.
    genre_loops = {
      "EDM / folk-house": [[0,5,3,4], [0,3,5,4], [5,3,0,4]],
      "Country": [[0,4,0,5], [0,5,3,4], [0,3,4,0]],
      "Jazz (swing)": [[1,4,0,0], [1,4,0,5], [0,3,1,4]],
      "Lo-fi hip-hop": [[0,3,5,4], [1,4,0,3], [0,5,3,4]],
      "Ambient": [[0,3,5,1], [0,4,1,5], [0,5,3,1]],
      "Rock": [[0,3,4,0], [0,5,3,4], [0,3,5,4]],
    }
    if genre in genre_loops:
        loops = genre_loops[genre]
        degs = loops[prog_id % len(loops)] * (bars // 4)
    degs[-1] = 0
    labels = [CH[d]["label"] for d in degs]

    progress(0.06, "PQCA evolves the rhythmic pattern ...")
    lo, hi = BUSY[p["busy"]]
    min_on = max(1, round(slots * lo)); max_on = max(min_on, round(slots * hi))
    rhythm = quantum_rhythm(bars, slots, min_on, max_on)
    # Jazz: retain the PQCA-generated onsets but gently favour syncopated offbeats.
    if genre in ("Jazz (swing)", "Lo-fi hip-hop"):
        rr = np.random.default_rng(CFG["seed"] if CFG["seed"] is not None else None)
        for bar_pattern in rhythm:
            if slots >= 8:
                probability = 0.38 if genre == "Jazz (swing)" else 0.22
                for offbeat in (1, 3, 5, 7):
                    if rr.random() < probability: bar_pattern[offbeat] = 1

    progress(0.10, "Generating Bell-pair accompaniment decisions ...")
    bell = bell_pairs(bars)
    assert all(a == b for a, b in bell)

    melody, log, demo = [], [], None
    prev_i, prev_dir = len(MEL) // 2, 0
    n_notes_total = sum(1 for bar in range(bars) for slot in range(slots) if rhythm[bar][slot] and bar < bars - 1) + 1
    done = 0
    for bar in range(bars):
        tones = CH[degs[bar]]["tones"]
        last_bar = (bar == bars - 1)
        onsets = [0] if last_bar else [slot for slot in range(slots) if rhythm[bar][slot]]
        for n_i, slot in enumerate(onsets):
            strong = slot in ts["strong"]
            force = final_i if last_bar else None
            idx, qc, k, pb, used = quantum_note(MEL, NQ, prev_i, prev_dir, tones, strong, force, genre)
            if demo is None and strong and force is None and k > 0:
                demo = dict(probs=used, marked=[i for i, m in enumerate(MEL) if m % 12 in tones], label=labels[bar], qc=qc)
            log.append((bar, slot, strong, degs[bar], k, pb, idx))
            end = onsets[n_i + 1] if n_i + 1 < len(onsets) else slots
            melody.append((bar * slots + slot, end - slot, MEL[idx], 84 if strong else 72))
            prev_dir, prev_i = idx - prev_i, idx
            done += 1
            if done % 4 == 0: progress(0.12 + 0.80 * done / max(1, n_notes_total), f"Measuring note {done} / {n_notes_total} ...")

    bass, pad, arp, harmony, guitar, strings = [], [], [], [], [], []
    for bar in range(bars):
        ch = CH[degs[bar]]; t0 = bar * slots
        root = 36 + ch["root"]
        if root < 40: root += 12
        fifth = root + (6 if ch["dim"] else 7)
        voicing = sorted(57 + ((pc - 57) % 12) for pc in ch["tones"])
        q_mel, q_bass = bell[bar]
        if genre == "Jazz (swing)":
            # Four-to-the-bar walking bass + offbeat piano comping.
            bass_tones = ch["tones"]
            walk = [0, 1, 2, min(3, len(bass_tones)-1)]
            for beat, deg in enumerate(walk):
                pc = bass_tones[deg]; pitch = 36 + pc
                if pitch < 40: pitch += 12
                bass.append((t0 + 2*beat, 2, pitch, 76 if beat in (0,2) else 66))
            for beat in (1,3):
                for v in voicing: pad.append((t0 + 2*beat, 1, v, 48))
            # Sparse sax-like answer phrase at the end of every second bar.
            if bar % 2 == 1:
                for sl, ix in ((5,2),(7,1)):
                    guitar.append((t0+sl,1,voicing[ix % len(voicing)]+12,46))
        elif genre == "Country":
            # Alternating root/fifth bass on quarter notes; acoustic boom-chick strums.
            for beat in range(4):
                bass.append((t0+2*beat,2,root if beat%2==0 else fifth,76 if beat%2==0 else 68))
            for sl in (0,2,4,6):
                chord = voicing if sl in (0,4) else voicing[1:]
                for v in chord: guitar.append((t0+sl,1,v+12,62 if sl in (0,4) else 48))
            # Occasional fiddle/steel-style fill between vocal phrases.
            if bar % 2 == 1:
                guitar += [(t0+5,1,root+19,54),(t0+7,1,root+21,48)]
        elif genre == "EDM / folk-house":
            # Acoustic-guitar-like offbeat strums over a four-on-the-floor bass pulse.
            for sl in range(0,slots,2): bass.append((t0+sl,1,root,88 if sl==0 else 76))
            for sl in (0,2,4,6):
                # alternating down/up strum velocity gives a recognisable folk-house motion
                for v in voicing: guitar.append((t0+sl,1,v+12,72 if sl in (0,4) else 54))
            for sl in (1,3,5,7):
                for v in voicing[1:]: guitar.append((t0+sl,1,v+12,42))
            # A bright, repeating synth arpeggio locks to the chord.
            order = [0,2,1,2]
            for sl in range(slots): arp.append((t0+sl,1,voicing[order[sl%4] % len(voicing)]+12,52))
            # Pads enter gradually, then open up in the drop.
            if bar >= int(bars*0.20):
                gain = 34 if bar < int(bars*0.70) else 58
                pad += [(t0,slots,v,gain) for v in voicing]
        elif genre == "Lo-fi hip-hop":
            # Dusty electric-piano chord stabs; swung sparse top-note motif.
            for sl in (0,5):
                for v in voicing: pad.append((t0+sl,2,v,42 if sl==0 else 34))
            for beat in (0,2): bass.append((t0+2*beat,2,root,58))
            top = voicing[-1]
            for sl in (1,4,7): arp.append((t0+sl,1,top,28 if sl!=4 else 36))
        elif genre == "Ambient":
            # Slow evolving chord bed, with a restrained arpeggio that changes with harmony.
            pad += [(t0,slots,v,S["pad_vel"]) for v in voicing]
            bass.append((t0,slots,root,48))
            if bar % 2 == 0:
                for sl, ix in ((0,0),(3,1),(6,2)):
                    arp.append((t0+sl,2,voicing[ix % len(voicing)]+12,24))
        elif genre == "Rock":
            # Power-chord guitar hits and sustained root/fifth bass.
            power = [root+12, root+19]
            for sl in (0,2,4,6):
                vel = 100 if sl in (0,4) else 78
                for v in power: guitar.append((t0+sl,1,v,vel))
            bass += [(t0,2,root,94),(t0+4,2,root,86)]
            pad += [(t0,slots,root+12,22)]
        else:
            # Pop: stable bass roots, chord pad and a regular eighth-note hook.
            pattern = ts["sync"] if q_bass else ts["plain"]
            for sl,dur,who in pattern: bass.append((t0+sl,dur,root if who=="r" else fifth,78))
            pad += [(t0,slots,v,S["pad_vel"]) for v in voicing]
            if S["arp"]:
                order=[0,1,2,1] if len(voicing)==3 else [0,1,2,3,2,1]
                for sl in range(slots): arp.append((t0+sl,1,voicing[order[sl%len(order)]],S["arp_vel"]))
        if bar == bars-1:
            bass.append((t0,slots,root,84))
            pad += [(t0,slots,v,max(35,S["pad_vel"])) for v in voicing]

    for (start, dur, m, vel) in melody:
        bar = start // slots
        if bell[bar][0] and bar != bars - 1:
            tones = CH[degs[bar]]["tones"]
            if m % 12 in tones:
                below = [m - d for d in (3, 4, 5) if (m - d) % 12 in tones]
                if below: harmony.append((start, dur, below[0], 62))

    drums = make_drum_pattern(style["drums"], bars, slots)
    progress(0.95, "Composition finished.")
    return dict(p=p, genre=genre, style=style, sc=sc, ts=ts, S=S, CH=CH, MEL=MEL, NQ=NQ, tonic=tonic,
                slots=slots, slot_sec=slot_sec, bars=bars, total=total, bits=bits, prog_id=prog_id,
                degs=degs, labels=labels, rhythm=rhythm, min_on=min_on, max_on=max_on, bell=bell,
                melody=melody, harmony=harmony, bass=bass, pad=pad, arp=arp, guitar=guitar, strings=strings, drums=drums,
                log=log, demo=demo, final_i=final_i)

print("Genre-aware hybrid composer defined.")

Genre-aware hybrid composer defined.


## 6. Sound (FM synthesis) & MIDI export

In [ ]:
def mtof(m): return 440.0 * 2 ** ((m - 69) / 12)

def env(n, a, r, curve=0.0):
    n = int(n); na = min(max(1, int(a * SR)), n); nr = min(max(1, int(r * SR)), n)
    e = np.ones(n); e[:na] = np.linspace(0, 1, na); e[-nr:] *= np.linspace(1, 0, nr)
    if curve: e *= np.exp(-curve * np.arange(n) / SR)
    return e

def voice_fm(f, dur, vel, S):
    n = int((dur + S["mel_release"] + 0.05) * SR); t = np.arange(n) / SR
    idx = S["idx0"] * np.exp(-S["idx_decay"] * t) + S["idx_base"]
    y = np.sin(2*np.pi*f*t + idx * np.sin(2*np.pi*f*2*t)) * env(n, S["mel_attack"], S["mel_release"], curve=S["mel_curve"])
    y += 0.25 * np.sin(2*np.pi*f*0.5*t) * env(n, 0.01, 0.3, curve=1.5)
    return y * (vel / 127) * 0.55

def voice_jazz_lead(f, dur, vel, S):
    """Sax-like additive lead with breathy harmonics and gentle vibrato (synthetic, not a sample)."""
    n = int((dur + 0.24) * SR); t = np.arange(n) / SR
    vib = 1.0 + 0.004 * np.sin(2*np.pi*5.2*t)
    phase = 2*np.pi*f*t*vib
    y = (np.sin(phase) + 0.48*np.sin(2*phase) + 0.22*np.sin(3*phase) + 0.10*np.sin(4*phase))
    y *= env(n, 0.035, 0.18, curve=0.45)
    return y * (vel / 127) * 0.34

def voice_bass(f, dur, vel, S):
    n = int((dur + 0.1) * SR); t = np.arange(n) / SR
    y = np.sin(2*np.pi*f*t) + 0.35*np.sin(2*np.pi*2*f*t) + 0.12*np.sin(2*np.pi*3*f*t)
    return y * env(n, 0.01, 0.12, curve=0.8) * (vel / 127) * 0.55

def voice_pad(f, dur, vel, S):
    n = int((dur + 0.5) * SR); t = np.arange(n) / SR
    y = sum(np.sin(2*np.pi*f*(1+d)*t) for d in (-0.003, 0.0, 0.003)) / 3 + 0.3*np.sin(2*np.pi*2*f*t)
    return y * env(n, 0.25, 0.45) * (vel / 127) * S["pad_gain"]

def voice_pluck(f, dur, vel, S):
    n = int((dur + 0.25) * SR); t = np.arange(n) / SR
    y = np.sin(2*np.pi*f*t) + 0.4*np.sin(2*np.pi*2*f*t)
    return y * env(n, 0.003, 0.1, curve=6.0) * (vel / 127) * 0.4

def voice_drum(pitch, dur, vel, seed=0):
    """Small synthesized GM-style kick/snare/hat/ride palette."""
    n = max(64, int(dur * SR)); t = np.arange(n) / SR
    rng = np.random.default_rng(int(seed) % (2**32 - 1))
    noise = rng.standard_normal(n)
    if pitch == 36:  # kick
        freq = 105 * np.exp(-t * 22) + 42
        phase = 2*np.pi*np.cumsum(freq) / SR
        y = np.sin(phase) * np.exp(-t * 13)
    elif pitch in (37, 38, 39):  # side-stick / snare / clap
        y = (0.72*noise + 0.28*np.sin(2*np.pi*185*t)) * np.exp(-t * (18 if pitch == 38 else 25))
    else:  # closed hat (42) or ride (51)
        y = noise * np.exp(-t * (48 if pitch == 42 else 9))
        y = np.concatenate([[0.0], np.diff(y)])
    return y * (vel / 127) * (0.34 if pitch in (36,38,39) else 0.18)


def voice_acoustic_guitar(f, dur, vel, S):
    """Short plucked/strummed acoustic-guitar approximation (not a sample)."""
    n=max(64,int((dur+0.18)*SR)); t=np.arange(n)/SR
    body=(np.sin(2*np.pi*f*t)+0.42*np.sin(2*np.pi*2*f*t)+0.18*np.sin(2*np.pi*3*f*t))
    body += 0.10*np.sin(2*np.pi*4.03*f*t)
    return body*env(n,0.002,0.16,curve=5.5)*(vel/127)*0.30

def voice_epiano(f, dur, vel, S):
    """Rounded tine-like electric piano approximation for lo-fi."""
    n=max(64,int((dur+0.30)*SR)); t=np.arange(n)/SR
    y=np.sin(2*np.pi*f*t)*np.exp(-t*2.8)
    y += 0.48*np.sin(2*np.pi*2.01*f*t)*np.exp(-t*5.5)
    y += 0.16*np.sin(2*np.pi*3.98*f*t)*np.exp(-t*9.0)
    return y*env(n,0.004,0.22)*(vel/127)*0.34

def voice_ambient(f, dur, vel, S):
    n=max(64,int((dur+1.2)*SR)); t=np.arange(n)/SR
    slow=0.5+0.5*np.sin(2*np.pi*0.12*t)
    y=(np.sin(2*np.pi*f*t)+0.45*np.sin(2*np.pi*1.997*f*t)+0.22*np.sin(2*np.pi*3.01*f*t))*slow
    return y*env(n,0.35,0.85)*(vel/127)*0.28

def voice_fiddle(f, dur, vel, S):
    n=max(64,int((dur+0.25)*SR)); t=np.arange(n)/SR
    vib=1+0.003*np.sin(2*np.pi*5.5*t)
    y=np.sin(2*np.pi*f*t*vib)+0.45*np.sin(4*np.pi*f*t*vib)+0.20*np.sin(6*np.pi*f*t*vib)
    return y*env(n,0.025,0.18)*(vel/127)*0.30

def voice_rock_guitar(f, dur, vel, S):
    n=max(64,int((dur+0.16)*SR)); t=np.arange(n)/SR
    raw=np.sin(2*np.pi*f*t)+0.55*np.sin(4*np.pi*f*t)+0.22*np.sin(6*np.pi*f*t)
    driven=np.tanh(S.get("drive",1.8)*raw)
    return driven*env(n,0.002,0.13,curve=2.8)*(vel/127)*0.38

def add_vinyl_texture(mix, amount, seed=23):
    if amount <= 0: return mix
    rng=np.random.default_rng(seed)
    noise=rng.normal(0,1,len(mix))
    # low-level filtered crackle/noise; deliberately subtle
    noise=sosfilt(butter(1,3500,fs=SR,output="sos"),noise)
    noise=noise/(np.max(np.abs(noise))+1e-9)
    return mix + amount*noise

def _slot_seconds(slot, slot_sec, swing):
    # Eighth-note offbeats move from 1/2 beat to 2/3 beat at 2:1 swing.
    whole = int(slot)
    return whole * slot_sec + ((whole % 2) * (2.0 * (swing - 0.5) * slot_sec) if swing > 0.5 else 0.0)

def render_audio(piece):
    S, slot_sec = piece["S"], piece["slot_sec"]
    swing=float(piece.get("style",{}).get("swing",0.5))
    out=np.zeros(int((piece["total"]+4.0)*SR))
    genre=piece.get("genre","")
    lead_map={"Jazz (swing)":voice_jazz_lead,"Country":voice_fiddle,
              "Lo-fi hip-hop":voice_epiano,"Ambient":voice_ambient,
              "Rock":voice_rock_guitar,"EDM / folk-house":voice_bright if 'voice_bright' in globals() else voice_pluck}
    lead_voice=lead_map.get(genre,voice_fm)
    pad_voice=voice_ambient if genre=="Ambient" else (voice_epiano if genre=="Lo-fi hip-hop" else voice_pad)
    guitar_voice=voice_rock_guitar if genre=="Rock" else voice_acoustic_guitar
    tracks=[(piece["pad"],pad_voice),(piece["bass"],voice_bass),(piece["arp"],voice_pluck),
            (piece["harmony"],voice_fm),(piece["melody"],lead_voice),(piece.get("guitar",[]),guitar_voice)]
    for notes,vf in tracks:
        for slot,dur,pitch,vel in notes:
            start_sec=_slot_seconds(slot,slot_sec,swing)
            end_sec=_slot_seconds(slot+dur,slot_sec,swing)
            actual_dur=max(0.025,end_sec-start_sec)*0.98
            y=vf(mtof(pitch),actual_dur,vel,S)
            i0=int(start_sec*SR); count=max(0,min(len(y),len(out)-i0))
            if count: out[i0:i0+count]+=y[:count]
    for slot,dur,pitch,vel in piece.get("drums",[]):
        start_sec=_slot_seconds(slot,slot_sec,swing)
        y=voice_drum(pitch,max(0.04,dur*slot_sec),vel,seed=slot*97+pitch*13)
        i0=int(start_sec*SR); count=max(0,min(len(y),len(out)-i0))
        if count: out[i0:i0+count]+=y[:count]
    # Genre-specific production: lo-fi texture and subtle pitch wobble;
    # ambient long tails; EDM sidechain-like pad ducking on kick pulses.
    if genre=="EDM / folk-house" and len(piece.get("pad",[])):
        # duck the sustained pad on each quarter-note kick to create pulse
        duck=np.ones(len(out)); beat=int((2*slot_sec)*SR); decay=int(0.16*SR)
        for slot,_,pitch,_ in piece.get("drums",[]):
            if pitch==36:
                i=int(_slot_seconds(slot,slot_sec,swing)*SR)
                duck[i:min(len(duck),i+decay)] = np.linspace(0.52,1.0,max(0,min(decay,len(duck)-i)))
        # Apply ducking to the overall mix gently; transient drums remain audible.
        out *= duck
    if genre=="Lo-fi hip-hop":
        # subtle wow/flutter on the rendered mix via slow amplitude modulation
        t=np.arange(len(out))/SR
        out *= (1.0+S.get("wow",0.0)*np.sin(2*np.pi*0.23*t))
    rng=np.random.default_rng(1); n_ir=int(1.2*SR)
    ir=rng.standard_normal(n_ir)*np.exp(-S["rt"]*np.arange(n_ir)/SR)
    ir[0]=0; ir/=np.abs(ir).sum()*0.12
    wet=fftconvolve(out,ir)[:len(out)]
    mix=(1-S["reverb"])*out+S["reverb"]*wet
    if S.get("lowpass"): mix=sosfilt(butter(2,S["lowpass"],fs=SR,output="sos"),mix)
    mix=add_vinyl_texture(mix,S.get("vinyl",0.0),seed=31)
    peak=np.max(np.abs(mix))
    return mix/peak*0.9 if peak else mix


def wav_bytes(audio):
    buf = io.BytesIO(); wavfile.write(buf, SR, (audio * 32767).astype(np.int16)); return buf.getvalue()

def midi_bytes(piece, filename="quantum_melody.mid"):
    ts, bpm = piece["ts"], piece["p"]["bpm"]
    mid = MidiFile(type=1, ticks_per_beat=480)
    swing = float(piece.get("style", {}).get("swing", 0.5))
    def slot_tick(slot):
        whole = int(slot)
        return whole * 240 + (int(round(480 * (swing - 0.5))) if whole % 2 and swing > 0.5 else 0)
    genre = piece.get("genre", "")
    lead_program = 65 if genre == "Jazz (swing)" else (40 if genre == "Country" else (30 if genre == "Rock" else (4 if genre == "Lo-fi hip-hop" else 80)))
    pad_program = 4 if genre in ("Lo-fi hip-hop", "Ambient") else 0
    bass_program = 32 if genre in ("Jazz (swing)", "Country") else 33
    guitar_program = 25 if genre in ("Country", "EDM / folk-house") else (29 if genre == "Rock" else 24)
    spec = [("melody", piece["melody"], 0, lead_program), ("harmony", piece["harmony"], 4, lead_program),
            ("pad", piece["pad"], 1, pad_program), ("bass", piece["bass"], 2, bass_program),
            ("arp", piece["arp"], 3, 46), ("guitar", piece.get("guitar", []), 5, guitar_program),
            ("drums", piece.get("drums", []), 9, None)]
    for k, (nm, notes, ch, prog) in enumerate(spec):
        tr = MidiTrack(); mid.tracks.append(tr)
        tr.append(MetaMessage("track_name", name=nm, time=0))
        if k == 0:
            tr.append(MetaMessage("set_tempo", tempo=bpm2tempo(bpm), time=0))
            tr.append(MetaMessage("time_signature", numerator=ts["num"], denominator=ts["den"], time=0))
        if prog is not None: tr.append(Message("program_change", channel=ch, program=prog, time=0))
        ev = []
        for slot, dur, pitch, vel in notes:
            start = slot_tick(slot); end = slot_tick(slot + dur)
            ev.append((start, 1, "note_on", pitch, vel))
            ev.append((max(start+1, end-6), 0, "note_off", pitch, 0))
        ev.sort(key=lambda e: (e[0], e[1]))
        last = 0
        for tick_pos, _, typ, pitch, vel in ev:
            tr.append(Message(typ, channel=ch, note=pitch, velocity=vel, time=max(0, tick_pos-last))); last=tick_pos
    mid.save(filename)
    with open(filename, "rb") as f: return f.read()

print("Genre-aware synthesis, drums, swing timing and MIDI export defined.")

Genre-aware synthesis, drums, swing timing and MIDI export defined.


## 7. Evaluation, plots and explanations

In [ ]:
def evaluate(piece):
    """Evaluate musical coherence plus measurable genre-feature alignment."""
    MEL, CH, log, melody = piece["MEL"], piece["CH"], piece["log"], piece["melody"]
    strong = [(MEL[idx], CH[d]["tones"]) for (bar, slot, st, d, k, pb, idx) in log if st]
    rb = np.random.default_rng(0)
    ct_ours = float(np.mean([m % 12 in t for m, t in strong])) if strong else float("nan")
    ct_base = float(np.mean([MEL[rb.integers(len(MEL))] % 12 in t for _, t in strong])) if strong else float("nan")
    idxs = [MEL.index(m) for _, _, m, _ in melody]
    leaps = np.abs(np.diff(idxs)); base_leaps = np.abs(np.diff(rb.integers(0, len(MEL), len(idxs))))
    scale_pcs = {(piece["tonic"] + i) % 12 for i in piece["sc"]["melody"]}
    gk = [l[4] for l in log if l[2]]; gp = [l[5] for l in log if l[2]]
    # Rhythmic metrics are computed from the generated onset grid, before swing rendering.
    onset_slots = [slot for bar, slot, *_ in log]
    offbeat_ratio = float(np.mean([slot % 2 == 1 for slot in onset_slots])) if onset_slots else 0.0
    density = float(np.mean([sum(row) / len(row) for row in piece["rhythm"]]))
    repeated_ratio = float(np.mean(np.diff(idxs) == 0)) if len(idxs) > 1 else 0.0
    drum_pitches = [e[2] for e in piece.get("drums", [])]
    drum_counts = {int(x): int(drum_pitches.count(x)) for x in sorted(set(drum_pitches))}
    genre = piece.get("genre", "")
    # Transparent rule checks, not an ML genre classifier or subjective listening score.
    guitar_events=len(piece.get("guitar",[]))
    if genre == "Jazz (swing)":
        genre_checks = {"Swing timing enabled": piece["style"]["swing"] >= 0.60,
                        "Ride cymbal present": 51 in drum_pitches,
                        "Walking bass generated": piece["S"]["bass"] == "walking",
                        "Piano comping stabs": len(piece["pad"]) >= piece["bars"],
                        "Seventh chords enabled": piece["p"]["sevenths"]}
    elif genre == "Country":
        genre_checks = {"Alternating bass generated": len(piece["bass"]) >= piece["bars"]*3,
                        "Acoustic strumming track": guitar_events > 0,
                        "Train-beat percussion": 42 in drum_pitches and 38 in drum_pitches,
                        "Major pentatonic vocabulary": "pentatonic" in piece["p"]["scale"].lower()}
    elif genre == "EDM / folk-house":
        kick_slots=[e[0] % piece["slots"] for e in piece.get("drums",[]) if e[2]==36]
        genre_checks={"Four-on-the-floor kick": all(s in kick_slots for s in (0,2,4,6)),
                      "Acoustic strum layer": guitar_events>0,
                      "Pulsing bass": len(piece["bass"]) >= piece["bars"]*3,
                      "Build/drop pad layer": len(piece["pad"])>0}
    elif genre == "Lo-fi hip-hop":
        genre_checks = {"Subtle vinyl texture": piece["S"].get("vinyl",0)>0,
                        "Electric-piano lead": piece["S"].get("lead")=="epiano",
                        "Swing timing": piece["style"]["swing"]>0.52,
                        "Sparse rhythm preset": piece["p"]["busy"]=="Sparse"}
    elif genre == "Ambient":
        genre_checks = {"Long evolving pad": piece["S"]["mel_attack"] >= 0.10,
                        "Sustained chord bed": len(piece["pad"]) >= piece["bars"]*3,
                        "No drum kit": len(drum_pitches)==0,
                        "Slow harmonic arpeggio": len(piece["arp"])>0}
    elif genre == "Rock":
        genre_checks = {"Rock drum backbeat": 38 in drum_pitches,
                        "Kick drum present": 36 in drum_pitches,
                        "Busy rhythm preset": piece["p"]["busy"] == "Busy"}
    elif genre == "Folk":
        genre_checks = {"Major pentatonic scale": "pentatonic" in piece["p"]["scale"].lower(),
                        "Waltz-like meter or steady 4/4": piece["p"]["ts"] in ("3/4", "4/4")}
    else:
        genre_checks = {"Moderate pop tempo": 90 <= piece["p"]["bpm"] <= 125,
                        "Pop backbeat present": 38 in drum_pitches,
                        "Moderate rhythm preset": piece["p"]["busy"] == "Medium"}
    genre_score = 100 * sum(bool(v) for v in genre_checks.values()) / max(1, len(genre_checks))
    return dict(ct_ours=ct_ours, ct_base=ct_base, leaps=leaps, base_leaps=base_leaps,
                mean_leap=float(leaps.mean()) if len(leaps) else 0.0,
                mean_base_leap=float(base_leaps.mean()) if len(base_leaps) else 0.0,
                step_ours=float(np.mean(leaps <= 1)) if len(leaps) else 0.0,
                step_base=float(np.mean(base_leaps <= 1)) if len(base_leaps) else 0.0,
                in_scale=all(m % 12 in scale_pcs for _, _, m, _ in melody),
                ends_on_tonic=(melody[-1][2] % 12 == piece["tonic"]),
                long_enough=(piece["total"] >= 20), n_strong=len(strong),
                mean_p=float(np.mean(gp)) if gp else float("nan"),
                grover_counts={int(a): int(b) for a, b in zip(*np.unique(gk, return_counts=True))} if gk else {},
                genre_checks=genre_checks, genre_score=genre_score,
                density=density, offbeat_ratio=offbeat_ratio, repeated_ratio=repeated_ratio,
                drum_counts=drum_counts)

def fig_piano_roll(piece):
    fig, ax = plt.subplots(figsize=(13, 4.6))
    ss = piece["slot_sec"]; slots = piece["slots"]
    for notes, col in [(piece["bass"], "#1f77b4"), (piece["pad"], "#c7e9c0"), (piece["arp"], "#9ecae1"),
                       (piece["harmony"], "#ff9896"), (piece["melody"], "#d62728")]:
        for s, d, pitch, v in notes:
            ax.add_patch(Rectangle((s * ss, pitch - 0.4), d * ss * 0.95, 0.8, color=col))
    for b in range(piece["bars"] + 1): ax.axvline(b * slots * ss, color="gray", lw=.4)
    for b, c in enumerate(piece["labels"]): ax.text(b * slots * ss + .05, 35.5, c, fontsize=8)
    ax.set_xlim(0, piece["total"]); ax.set_ylim(35, 92)
    ax.set_xlabel("seconds"); ax.set_ylabel("MIDI pitch")
    ax.set_title("Quantum composition  -  red: melody, pink: harmony voice, blue: bass, green: pad")
    plt.tight_layout(); return fig

def fig_eval(ev):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3))
    ax[0].bar(["quantum (ours)", "random baseline"], [ev["ct_ours"]*100, ev["ct_base"]*100], color=["#d62728", "#999"])
    ax[0].set_ylabel("% chord tones on strong beats"); ax[0].set_ylim(0, 100)
    top = int(max(ev["leaps"].max(), ev["base_leaps"].max())) + 2
    ax[1].hist([ev["leaps"], ev["base_leaps"]], bins=range(0, top), label=["ours", "baseline"], color=["#d62728", "#999"])
    ax[1].set_xlabel("melodic leap (scale steps)"); ax[1].legend(); plt.tight_layout(); return fig

def grover_demo_data(piece, shots=4000):
    d = piece["demo"]
    if d is None: return None
    qc, k, p = amplitude_amplification_circuit(d["probs"], d["marked"])
    res = run_circuit(qc, shots=shots)
    n = len(piece["MEL"])
    counts = np.bincount([int(s, 2) for s in res], minlength=2**piece["NQ"])[:n] / shots
    return dict(qc=qc, k=k, p=p, counts=counts, measured=float(counts[d["marked"]].sum()))

def fig_grover(piece, gd):
    d, MEL = piece["demo"], piece["MEL"]; n = len(MEL); x = np.arange(n); wd = 0.4
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.bar(x - wd/2, d["probs"][:n], wd, label="Markov distribution (QuPoly)")
    ax.bar(x + wd/2, gd["counts"], wd, label="after amplitude amplification")
    for i in d["marked"]: ax.axvspan(i-.5, i+.5, color="gold", alpha=.15)
    ax.set_xticks(x); ax.set_xticklabels([name(m) for m in MEL], rotation=45); ax.set_ylabel("probability")
    ax.set_title(f"Strong beat over {d['label']} - gold = chord tones"); ax.legend(); plt.tight_layout(); return fig

def explain_html(piece, ev):
    p, labels = piece["p"], piece["labels"]
    on = [sum(r) for r in piece["rhythm"]]
    gc = ev["grover_counts"]
    items = [
      f"<b>Chord progression (Hadamard QRNG).</b> 2 qubits in superposition were measured and gave <code>{piece['bits']}</code> "
      f"= progression #{piece['prog_id']}: <b>{' – '.join(labels[:4])}</b>, repeated {piece['bars']//4}× and ending on the tonic chord.",
      f"<b>Rhythm (PQCA).</b> {piece['slots']} qubits (one per eighth-note slot) evolved bar by bar; the measured bits are the note onsets. "
      f"Onsets per bar ranged from {min(on)} to {max(on)} (allowed {piece['min_on']}–{piece['max_on']} for ‘{p['busy']}’).",
      f"<b>Melody (quantum Markov chain).</b> {len(piece['melody'])} notes, each sampled from a {piece['NQ']}-qubit state whose amplitudes encode the "
      f"musical rules ({len(piece['MEL'])} allowed pitches).",
      f"<b>Harmony on strong beats (Grover).</b> {ev['n_strong']} strong beats used amplitude amplification "
      f"(iterations used: {gc}). Chord-tone probability starts at ≈{ev['mean_p']:.2f} and is boosted to ≈0.97 in theory; "
      f"in this piece <b>{ev['ct_ours']*100:.0f}%</b> of strong-beat notes are chord tones vs {ev['ct_base']*100:.0f}% for a random baseline.",
      f"<b>Entanglement (Bell pairs).</b> {sum(1 for a, b in piece['bell'] if a)} of {piece['bars']} bars got a harmony voice and the same bars got the "
      f"syncopated bass – both qubits are perfectly correlated, so melody and bass ‘agree’.",
    ]
    return "<ol style='line-height:1.5'>" + "".join(f"<li>{i}</li>" for i in items) + "</ol>"

def draw_circuit(qc, out_widget, caption=None):
    with out_widget:
        if caption: display(HTML(f"<p style='margin:6px 0'>{caption}</p>"))
        try:
            fig = qc.draw("mpl", fold=-1); display(fig); plt.close(fig)
        except Exception:
            print(qc.draw("text"))

print("Evaluation + plotting helpers defined.")

Evaluation + plotting helpers defined.


## 8. Dashboard
Press **🎼 Compose**. Choosing a *Style preset* fills in sensible defaults (tempo, scale, busyness, number of pitches, 7th chords, sound) – every control can still be changed afterwards.

In [ ]:
HOW_IT_WORKS = """
<div style='line-height:1.55'>
<h3>How the quantum composer works</h3>
<table style='border-collapse:collapse'>
<tr><th style='text-align:left;padding:4px 12px'>Musical job</th><th style='text-align:left;padding:4px 12px'>Quantum idea</th><th style='text-align:left;padding:4px 12px'>What the circuit does</th></tr>
<tr><td style='padding:4px 12px'>Chord progression</td><td style='padding:4px 12px'>Hadamard QRNG</td><td style='padding:4px 12px'>H on 2 qubits → measure → one of 4 chord loops</td></tr>
<tr><td style='padding:4px 12px'>Rhythm</td><td style='padding:4px 12px'>Partitioned Quantum Cellular Automaton</td><td style='padding:4px 12px'>CRX + H/CNOT on shifting qubit pairs; measured bits = note onsets, fed into the next bar</td></tr>
<tr><td style='padding:4px 12px'>Melody pitches</td><td style='padding:4px 12px'>Quantum Markov chain (QuPoly)</td><td style='padding:4px 12px'>Music rules → probabilities → amplitudes (StatePreparation) → measure</td></tr>
<tr><td style='padding:4px 12px'>Harmonic strong beats</td><td style='padding:4px 12px'>Grover / amplitude amplification</td><td style='padding:4px 12px'>Oracle flips chord-tone phases, diffusion boosts them: P = sin²((2k+1)θ)</td></tr>
<tr><td style='padding:4px 12px'>Accompaniment decisions</td><td style='padding:4px 12px'>Bell-pair entanglement</td><td style='padding:4px 12px'>H + CNOT: correlated bits control harmony-voice and bass-pattern choices</td></tr>
</table>
<p><b>What the dashboard controls:</b> key, scale and chords (the “rules of the game”), tempo, how many different pitches the melody may use
(= number of qubits: 6–8 notes → 3 qubits, 9–16 notes → 4 qubits), rhythmic busyness (limits on PQCA onsets), time signature, 7th chords and the sound style.
Genre presets now change transition preferences, harmonic loops, swing timing, bass/comping patterns, acoustic/electric guitar parts, drum arrangements and sound-design profiles. EDM/folk-house uses acoustic strums plus four-on-the-floor drums; Country uses alternating bass and train-beat percussion; Lo-fi adds electric-piano timbre and subtle vinyl texture; Ambient uses sustained evolving pads and slow arpeggios. The same circuit families are reused with genre-conditioned classical probability rules.</p>
<p><b>Honest limits:</b> circuits run on a simulator (<code>AerSimulator</code>); no quantum speed-up is claimed. The quantum part makes the <i>decisions</i>,
classical music theory defines which options exist.</p>
</div>
"""

def make_dashboard():
    sty = dict(description_width="150px"); lay = W.Layout(width="420px")
    w_style = W.Dropdown(options=list(STYLES), description="Genre / style", style=sty, layout=lay)
    w_key   = W.Dropdown(options=[(n, i) for i, n in enumerate(NOTE_NAMES)], value=9, description="Key (tonic)", style=sty, layout=lay)
    w_scale = W.Dropdown(options=list(SCALES), description="Scale / mode", style=sty, layout=lay)
    w_swing = W.FloatSlider(value=0.5, min=0.5, max=0.70, step=0.01, description="Swing feel", style=sty, layout=lay, continuous_update=False)
    w_bpm   = W.IntSlider(value=96, min=50, max=180, step=1, description="Tempo (BPM)", style=sty, layout=lay, continuous_update=False)
    w_notes = W.IntSlider(value=11, min=6, max=16, step=1, description="Pitches in melody", style=sty, layout=lay, continuous_update=False)
    w_busy  = W.ToggleButtons(options=list(BUSY), value="Medium", description="Rhythm busyness", style=dict(description_width="150px", button_width="80px"))
    w_secs  = W.IntSlider(value=30, min=20, max=90, step=5, description="Length (seconds, ≥ 20)", style=sty, layout=lay, continuous_update=False)
    w_ts    = W.Dropdown(options=list(TIMESIGS), value="4/4", description="Time signature", style=sty, layout=lay)
    w_sev   = W.Checkbox(value=False, description="Jazzy 7th chords", indent=False)
    w_rand  = W.Checkbox(value=False, description="New random piece every click", indent=False)
    w_seed  = W.IntText(value=7, description="Seed (repeatable)", style=sty, layout=W.Layout(width="260px"))
    w_circ  = W.Checkbox(value=True, description="Show quantum circuits", indent=False)
    w_expl  = W.Checkbox(value=True, description="Show explanations", indent=False)
    btn     = W.Button(description="🎼  Compose", button_style="primary", layout=W.Layout(width="200px", height="44px"))
    bar     = W.FloatProgress(value=0, min=0, max=1, description="", layout=W.Layout(width="420px"))
    status  = W.HTML("")

    def apply_style(_=None):                          # a genre = a bundle of defaults you can still override
        s = STYLES[w_style.value]
        w_bpm.value, w_scale.value, w_busy.value = s["bpm"], s["scale"], s["busy"]
        w_notes.value, w_ts.value, w_sev.value = s["notes"], s["ts"], s["sevenths"]
        w_swing.value = s["swing"]
    w_style.observe(apply_style, names="value"); apply_style()

    tabs_out = [W.Output() for _ in range(5)]
    tabs = W.Tab(children=tabs_out)
    for i, t in enumerate(["🎧 Listen", "🎹 Piano roll", "⚛️ Circuits", "📊 Evaluation", "📖 How it works"]): tabs.set_title(i, t)
    with tabs_out[4]: display(HTML(HOW_IT_WORKS))

    def collect():
        return dict(genre=w_style.value, key=w_key.value, scale=w_scale.value, bpm=w_bpm.value, notes=w_notes.value, busy=w_busy.value,
                    seconds=w_secs.value, ts=w_ts.value, sevenths=w_sev.value, swing=w_swing.value, random_seed=w_rand.value, seed=w_seed.value,
                    sound=STYLES[w_style.value]["sound"], show_circuits=w_circ.value, show_expl=w_expl.value)

    def on_compose(_=None):
        btn.disabled = True; status.value = ""
        for o in tabs_out[:4]: o.clear_output()
        try:
            p = collect()
            def cb(f, msg=""): bar.value = f; status.value = f"<i>{msg}</i>"
            piece = compose(p, cb)
            audio = render_audio(piece); ev = evaluate(piece)
            wav, mid = wav_bytes(audio), midi_bytes(piece)
            with open("quantum_melody.wav", "wb") as f: f.write(wav)

            def link(data, fname, mime, label):
                b = base64.b64encode(data).decode()
                return (f"<a download='{fname}' href='data:{mime};base64,{b}' style='margin-right:14px;padding:6px 12px;"
                        f"border:1px solid #888;border-radius:6px;text-decoration:none'>{label}</a>")
            checks = [("Length ≥ 20 s", ev["long_enough"], f"{piece['total']:.1f} s, {piece['bars']} bars"),
                      ("All notes in scale", ev["in_scale"], ""), ("Ends on tonic", ev["ends_on_tonic"], "")]
            chips = " ".join(f"<span style='padding:3px 8px;border-radius:10px;background:{'#d4edda' if ok else '#f8d7da'};color:#222'>"
                             f"{'✔' if ok else '✘'} {t} {('(' + d + ')') if d else ''}</span>" for t, ok, d in checks)
            info = (f"<p><b>{NOTE_NAMES[p['key']]} {p['scale'].split(' (')[0]}</b> · {p['bpm']} BPM · {p['ts']} · "
                    f"{len(piece['MEL'])} pitches ({piece['NQ']} qubits) · progression <b>{' – '.join(piece['labels'][:4])}</b></p><p>{chips}</p>")
            with tabs_out[0]:
                display(HTML(info)); display(Audio(audio, rate=SR))
                display(HTML("<p>" + link(wav, "quantum_melody.wav", "audio/wav", "⬇ WAV") + link(mid, "quantum_melody.mid", "audio/midi", "⬇ MIDI") + "</p>"))
                if p["show_expl"]: display(HTML("<h4>What the quantum computer decided</h4>" + explain_html(piece, ev)))
            with tabs_out[1]:
                fig = fig_piano_roll(piece); display(fig); plt.close(fig)
            if not p["show_circuits"]:
                with tabs_out[2]: print("Tick ‘Show quantum circuits’ and compose again.")
            else:
                ex = p["show_expl"]
                draw_circuit(qrng_circuit(2), tabs_out[2], "<b>1 · QRNG</b> – Hadamard on 2 qubits picks the chord progression." if ex else None)
                draw_circuit(pqca_circuit(piece["rhythm"][0]), tabs_out[2], "<b>2 · PQCA (bar 1)</b> – X arms the previous bar, CRX(π/4) keeps memory, H+CX creates new patterns." if ex else None)
                gd = grover_demo_data(piece)
                if gd is not None:
                    draw_circuit(gd["qc"], tabs_out[2], f"<b>3 · Melody note on a strong beat</b> – StatePreparation encodes the Markov probabilities; k = {gd['k']} Grover iteration(s) boost chord tones from {gd['p']:.2f} to {gd['measured']:.2f} (measured)." if ex else None)
                    with tabs_out[2]:
                        gf = fig_grover(piece, gd); display(gf); plt.close(gf)
                draw_circuit(bell_circuit(), tabs_out[2], "<b>4 · Bell pair</b> – H + CNOT: correlated bits control accompaniment choices." if ex else None)
            with tabs_out[3]:
                gchecks = "".join(f"<li>{'✔' if ok else '✘'} {label}</li>" for label, ok in ev["genre_checks"].items())
                display(HTML(f"<h4>Genre validation: {piece['genre']} — {ev['genre_score']:.0f}% rule checks passed</h4><ul>{gchecks}</ul>"))
                display(HTML(f"<p>Rhythm density: {ev['density']*100:.1f}% · Offbeat onset ratio: {ev['offbeat_ratio']*100:.1f}% · Repeated-note ratio: {ev['repeated_ratio']*100:.1f}% · Drum events: {len(piece.get('drums', []))}</p>"))
                display(HTML(f"<ul><li>Strong-beat chord tones: <b>ours {ev['ct_ours']*100:.0f}%</b> vs random baseline {ev['ct_base']*100:.0f}%</li>"
                             f"<li>Mean melodic leap: ours {ev['mean_leap']:.2f} vs baseline {ev['mean_base_leap']:.2f} scale steps</li>"
                             f"<li>Stepwise motion: ours {ev['step_ours']*100:.0f}% vs baseline {ev['step_base']*100:.0f}%</li></ul>"))
                fe = fig_eval(ev); display(fe); plt.close(fe)
            bar.value = 1; status.value = "✅ Done – press play, or change a setting and compose again."
        except Exception:
            status.value = "<b style='color:#c00'>Something went wrong – details in the Listen tab.</b>"
            with tabs_out[0]: print(traceback.format_exc())
        finally:
            btn.disabled = False
    btn.on_click(on_compose)

    left  = W.VBox([w_style, w_key, w_scale, w_ts, w_sev])
    right = W.VBox([w_bpm, w_notes, w_secs, w_busy, w_swing])
    extras = W.HBox([w_rand, w_seed, w_circ, w_expl], layout=W.Layout(flex_flow="row wrap"))
    ui = W.VBox([W.HTML("<h2>🎼 Quantum Music Composer</h2>"), W.HBox([left, right], layout=W.Layout(flex_flow="row wrap")),
                 extras, W.HBox([btn, bar]), status, tabs])
    return ui, on_compose

## Original dashboard retained as a reference; upgraded dashboard is launched in the final cell.

## 9. Project summary and honest limitations

**Pipeline:** Quantum QRNG → harmonic-loop selection · PQCA → rhythmic onsets · quantum Markov sampling + Grover → melody · Bell pairs → correlated accompaniment choices · QAOA → genre-conditioned arrangement profile · FluidSynth/SoundFont → sampled-instrument WAV + MIDI.

**Hybrid design:** Classical music theory defines the candidate vocabulary and constraints. Quantum circuits sample and optimize decisions within that vocabulary. The genre layer assigns instrument roles, rhythmic behaviour, harmonic vocabulary, phrasing and section structure. Audio rendering is a separate stage.

**Research question:** Can genre-conditioned quantum probabilistic sampling and hybrid quantum-classical arrangement optimization improve measurable genre alignment and musical coherence relative to a classical rule-based baseline?

**Limitations:** Circuits run on AerSimulator in this notebook; this demonstrates quantum algorithm integration, not quantum advantage. The QAOA arrangement problem is deliberately small (3 decision qubits) so it can run in Colab. Rule checks and audio descriptors are not a trained genre classifier or a substitute for listening tests. SoundFont quality depends on the selected library and is sample-based, not a live recording.

**Evaluation:** Structural genre checks, harmonic alignment, rhythmic density/offbeat ratio, spectral centroid, low-frequency energy ratio, and a planned human listening test. Compare classical baseline, quantum-only composition, and quantum + QAOA arrangement using the same seeds and settings.


**Genre-specific melody update:** Pop, Jazz, EDM, Lo-fi, Ambient, Country and Rock now use distinct melodic probability priors and phrase behaviour. These priors shape quantum state-preparation amplitudes; Qiskit measurement samples each note and Grover remains available on eligible strong beats. The dashboard reports melodic interval, repetition, stepwise-motion and pitch-range metrics. This is a hybrid quantum/classical generator, not a claim of quantum advantage.


## 10. Sample-based instrument rendering (FluidSynth + SoundFont)

Run this setup cell once in Google Colab. It installs FluidSynth and a General MIDI SoundFont. If installation is unavailable, the notebook automatically falls back to the original FM renderer.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq fluidsynth fluid-soundfont-gm
!pip -q install pyfluidsynth


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1ubuntu0.1_amd64.deb ...
Unpacking libxtst6:amd64 (

### Updated visualisation and sound design
Grover's amplitude-amplification step is retained in the strong-beat melody path. The final dashboard now displays QRNG, PQCA, QuPoly state preparation, Grover, Bell-pair and QAOA circuits, plus their diagnostic plots. Ambient remains a selectable genre. Lo-fi includes a continuous band-limited rain/hiss layer in the WAV render; this atmospheric noise is an audio-production layer and therefore is not represented in the MIDI file.

## 11. Genre-conditioned quantum arrangement optimization

The following cell adds a 3-qubit, one-layer QAOA optimizer. It selects among eight candidate arrangement profiles. The objective is a small genre-weighted QUBO: missing genre-role decisions are penalized, with a few pairwise compatibility penalties. The classical optimizer tunes QAOA angles; measurement supplies a candidate profile. This is an educational hybrid demonstration, not evidence of quantum speedup.

The same cell fixes instrument-specific swing in MIDI timing, updates the four target genre arrangements, maps tracks to General MIDI programs, and renders through FluidSynth when available.


In [ ]:

# ============================================================
# 11. GENRE-CONDITIONED ARRANGEMENT + QAOA + SOUNDFONT ENGINE
# ============================================================
import os, glob, shutil, subprocess, tempfile
from scipy.optimize import minimize
from scipy.signal import welch
from qiskit.quantum_info import Statevector

# Keep the original quantum composer as the base generator.
# Capture the original implementations exactly once. This makes the extension cell
# safe to re-run without wrapping its own wrappers (which caused RecursionError).
if "_QMC_BASE_COMPOSE" not in globals():
    _QMC_BASE_COMPOSE = compose
if "_QMC_BASE_RENDER_AUDIO" not in globals():
    _QMC_BASE_RENDER_AUDIO = render_audio
if "_QMC_BASE_EVALUATE" not in globals():
    _QMC_BASE_EVALUATE = evaluate

# The jazz preset uses a major-key ii-V-I vocabulary by default.
# Users can still change the scale in the dashboard.
STYLES["Jazz (swing)"]["scale"] = "Major (bright)"
STYLES["Jazz (swing)"]["sevenths"] = True
STYLES["Jazz (swing)"]["swing"] = 0.667

# Explicitly named genre profiles. The values are starting points, not rigid definitions.
GENRE_FEATURES = {
    "Jazz (swing)": ["walking_bass", "piano_comping", "call_response"],
    "Pop / cinematic (default)": ["hook_motif", "backbeat", "guitar_support"],
    "EDM / folk-house": ["four_on_floor", "build_drop", "sidechain"],
    "Lo-fi hip-hop": ["loop_motif", "sparse_drums", "vinyl_texture"],
    "Ambient": ["evolving_pad", "sparse_arpeggio", "no_drums"],
}
GENRE_TARGET_WEIGHTS = {
    "Jazz (swing)": [4.0, 3.0, 2.0],
    "Pop / cinematic (default)": [3.0, 3.0, 2.0],
    "EDM / folk-house": [4.0, 3.0, 3.0],
    "Lo-fi hip-hop": [3.0, 4.0, 2.0],
    "Ambient": [4.0, 2.0, 3.0],
}

def transition_weights(mel, nq, prev_i, prev_dir, tones, strong,
                       genre="Pop / cinematic (default)"):
    """Classical music-theory constraints shape a probability distribution for quantum sampling."""
    n = len(mel)
    w = np.zeros(2**nq, dtype=float)
    chord_tones = {i for i, m in enumerate(mel) if m % 12 in tones}
    centre = (n - 1) / 2
    sigma = max(1.5, 3.0 * n / 11)
    for i, m in enumerate(mel):
        d = abs(i - prev_i)
        x = {0:0.22, 1:3.0, 2:2.1, 3:0.9, 4:0.48}.get(d, 0.15 if d == 5 else 0.05)
        if genre == "Jazz (swing)":
            if d in (2,3,4): x *= 1.35
            if not strong and i in chord_tones: x *= 1.35
        elif genre == "EDM / folk-house":
            if d == 0: x *= 1.35
            if d in (1,2): x *= 1.15
            if d >= 4: x *= 0.65
        elif genre == "Lo-fi hip-hop":
            if d <= 2: x *= 1.5
            if d >= 4: x *= 0.45
            if d == 0: x *= 1.25
        else:  # Pop
            if d == 0: x *= 1.35
            if d in (1,2): x *= 1.2
            if d >= 4: x *= 0.7
        if not strong and i in chord_tones: x *= 1.4
        if strong and i not in chord_tones:
            if any((m-c) % 12 in (1,11) for c in tones): x *= 0.6
        x *= math.exp(-((i-centre)**2)/(2*sigma**2))
        if abs(prev_dir) >= 3 and d in (1,2) and np.sign(i-prev_i) == -np.sign(prev_dir):
            x *= 2.2
        w[i] = x
    w = np.maximum(w, 1e-12)
    return w / w.sum()

def make_drum_pattern(genre, bars, slots):
    """GM percussion events: (absolute eighth-note slot, duration, pitch, velocity)."""
    templates = {
        "jazz": ([(i,51,68 if i in (0,4) else 48) for i in range(8)] +
                 [(2,37,34),(6,37,38)]),
        "pop": ([(0,36,92),(4,36,84),(2,38,96),(6,38,96)] +
                [(i,42,35 if i in (0,4) else 27) for i in (0,2,4,6)]),
        "edm": ([(i,36,118) for i in (0,2,4,6)] +
                [(2,39,88),(6,39,92)] + [(i,42,52) for i in range(8)]),
        "lofi": ([(0,36,62),(4,36,48),(2,37,43),(6,37,45)] +
                 [(i,42,22 if i%2==0 else 14) for i in range(8)]),
        "country": ([(0,36,88),(4,36,72),(2,38,70),(6,38,78)] +
                    [(i,42,34 if i%2==0 else 23) for i in range(8)]),
        "rock": ([(0,36,112),(4,36,100),(2,38,108),(6,38,112)] +
                 [(i,42,48) for i in range(8)]),
    }
    template = templates.get(genre, [])
    events = []
    for bar in range(bars):
        section = bar / max(1, bars)
        for slot, pitch, vel in template:
            if genre == "edm":
                # Intro: restrained; build: hats/snare increase; drop: full kick/clap.
                if section < 0.18 and pitch == 36 and slot != 0: continue
                if 0.50 <= section < 0.70 and pitch in (39,42): vel = min(127, int(vel*1.25))
                if section >= 0.70: vel = min(127, int(vel*1.08))
            mapped = int(round(slot*slots/8)) % slots
            events.append((bar*slots+mapped, 1, pitch, int(vel)))
        if genre == "edm" and 0.50 <= section < 0.70:
            # Short snare build in the final two bars before the drop.
            if bar >= int(bars*0.65):
                for s in (0,1,2,3,4,5,6,7):
                    mapped = int(round(s*slots/8)) % slots
                    events.append((bar*slots+mapped,1,38,42+s*5))
    return events

def _qaoa_costs(genre):
    """Three binary arrangement decisions and a small genre-weighted QUBO objective."""
    target = np.ones(3, dtype=int)
    weights = GENRE_TARGET_WEIGHTS.get(genre, [2.0,2.0,2.0])
    costs = np.zeros(8, dtype=float)
    for state in range(8):
        bits = np.array([(state >> q) & 1 for q in range(3)], dtype=int)
        missing = target - bits
        cost = float(np.dot(weights, missing))
        # Pairwise penalties: discourage incomplete combinations that break an arrangement role.
        if genre == "Jazz (swing)" and bits[0] and not bits[1]: cost += 1.0
        if genre == "EDM / folk-house" and bits[0] and not bits[2]: cost += 1.2
        if genre == "Lo-fi hip-hop" and bits[1] and not bits[0]: cost += 0.7
        costs[state] = cost
    return costs

def qaoa_arrangement(genre, shots=512):
    """
    One-layer QAOA over 3 binary arrangement choices.
    Classical outer-loop optimization tunes gamma/beta; the circuit samples candidate profiles.
    This is a small demonstrator, not a claim of quantum advantage.
    """
    n = 3
    costs = _qaoa_costs(genre)

    def state_circuit(gamma, beta):
        qc = QuantumCircuit(n)
        qc.h(range(n))
        qc.unitary(np.diag(np.exp(-1j*gamma*costs)), list(range(n)), label="QAOA cost phase")
        for q in range(n):
            qc.rx(2*beta, q)
        return qc

    def objective(theta):
        gamma, beta = float(theta[0]), float(theta[1])
        sv = Statevector.from_instruction(state_circuit(gamma,beta))
        probs = sv.probabilities()
        return float(np.dot(probs, costs))

    opt = minimize(objective, x0=np.array([0.7,0.35]), method="COBYLA",
                   options={"maxiter":35, "rhobeg":0.5, "tol":1e-3})
    gamma, beta = map(float, opt.x)
    qc = state_circuit(gamma,beta)
    measured = qc.copy()
    measured.measure_all()
    try:
        bitstrings = run_circuit(measured, shots=shots)
        counts = {}
        for b in bitstrings:
            state = int(b.replace(" ",""),2)
            counts[state] = counts.get(state,0)+1
        if not counts:
            raise RuntimeError("QAOA returned no measurement samples")
        selected = min(counts, key=lambda s: (costs[s], -counts[s]))
        method = "QAOA circuit + COBYLA"
    except Exception as exc:
        # Explicit fallback: keep the notebook usable, but report that the circuit sampling failed.
        selected = int(np.argmin(costs))
        counts = {selected:shots}
        method = "exact-cost fallback: " + str(exc)[:100]
    bits = [(selected >> q) & 1 for q in range(n)]
    profile = {name: bool(bit) for name,bit in zip(GENRE_FEATURES.get(genre,[]),bits)}
    return {
        "qc": measured, "gamma":gamma, "beta":beta, "expected_cost":objective([gamma,beta]),
        "costs":costs, "counts":counts, "selected_state":selected, "bits":bits,
        "features":profile, "method":method, "classical_optimum":float(np.min(costs))
    }

def _add_pop_hook(piece):
    """Repeat a short quantum-generated motif at chorus-like four-bar positions."""
    slots = piece["slots"]
    first_bar = [(s,d,m,v) for s,d,m,v in piece["melody"] if 0 <= s < slots]
    hook = []
    if not first_bar: return hook
    for bar in range(2, piece["bars"], 4):
        for s,d,m,v in first_bar:
            # Keep the motif recognisable; lightly lower velocity in the repeated hook.
            hook.append((bar*slots+s, d, m, max(38,int(v*0.88))))
    return hook

def _apply_genre_arrangement(piece, qaoa):
    genre = piece["genre"]
    f = qaoa["features"]
    slots = piece["slots"]
    bars = piece["bars"]
    piece["hook"] = []

    if genre == "Jazz (swing)":
        # Rebuild a genuine walking line: root, chord tone, fifth, chromatic approach to next root.
        walking = []
        for bar in range(bars):
            ch = piece["CH"][piece["degs"][bar]]
            nxt = piece["CH"][piece["degs"][min(bar+1,bars-1)]]
            root_pc = ch["root"]
            root = 36 + root_pc
            if root < 40: root += 12
            third = (ch["tones"][1]-root_pc) % 12
            fifth = (ch["tones"][2]-root_pc) % 12
            next_root = 36 + nxt["root"]
            if next_root < 40: next_root += 12
            approach = next_root - 1 if next_root > root else next_root + 1
            notes = [root, root+third, root+fifth, approach]
            for beat,pitch in enumerate(notes):
                walking.append((bar*slots+2*beat,2,int(pitch),76 if beat in (0,2) else 65))
        if f.get("walking_bass",True):
            piece["bass"] = walking
        # Piano comping is intentionally on selected beats; it is not globally swung.
        if not f.get("piano_comping",True):
            piece["pad"] = []
        # Short call-and-response answer phrases are separate from the continuous lead.
        if f.get("call_response",True):
            response = []
            for bar in range(1,bars,2):
                chord = piece["CH"][piece["degs"][bar]]
                tones = chord["tones"]
                for slot,ix in ((5,2),(7,1)):
                    pitch = 60 + tones[ix % len(tones)]
                    response.append((bar*slots+slot,1,pitch,42))
            piece["guitar"] = response
        else:
            piece["guitar"] = []

    elif genre == "Pop / cinematic (default)":
        if f.get("hook_motif",True):
            piece["hook"] = _add_pop_hook(piece)
        if f.get("guitar_support",True):
            # Light acoustic/electric chord punctuation, not an EDM-style continuous strum.
            guitar = []
            for bar in range(bars):
                t0 = bar*slots
                chord = piece["CH"][piece["degs"][bar]]
                voicing = sorted(60 + ((pc-60)%12) for pc in chord["tones"])
                for sl in (0,4):
                    for pitch in voicing:
                        guitar.append((t0+sl,1,pitch,38 if sl==0 else 32))
            piece["guitar"] = guitar

    elif genre == "EDM / folk-house":
        # EDM uses electronic roles, not the acoustic strum layer in the earlier prototype.
        piece["guitar"] = []
        if not f.get("four_on_floor",True):
            piece["drums"] = [e for e in piece["drums"] if not (e[2]==36 and e[0]%slots in (0,2,4,6))]
        if not f.get("build_drop",True):
            # Remove section-dependent snare build; retain the core dance beat.
            piece["drums"] = [e for e in piece["drums"] if e[2] != 38 or e[3] < 100]
        if not f.get("sidechain",True):
            piece["S"]["sidechain"] = False
        else:
            piece["S"]["sidechain"] = True

    elif genre == "Lo-fi hip-hop":
        # Sparse repeated electric-piano chord loop, soft drums, restrained bass and tape/vinyl texture.
        if f.get("loop_motif",True):
            piece["hook"] = _add_pop_hook(piece)
        if f.get("sparse_drums",True):
            piece["drums"] = [(s,d,p,min(v,65)) for s,d,p,v in piece["drums"]]
        if f.get("vinyl_texture",True):
            piece["S"]["vinyl"] = max(piece["S"].get("vinyl",0.0),0.012)
            piece["S"]["wow"] = max(piece["S"].get("wow",0.0),0.0015)

    elif genre == "Ambient":
        # Preserve the base composer's sustained chord bed and sparse evolving arpeggio.
        if not f.get("sparse_arpeggio", True):
            piece["arp"] = []
        if f.get("no_drums", True):
            piece["drums"] = []
        piece["S"]["reverb"] = max(piece["S"].get("reverb", 0.0), 0.62)
        piece["S"]["rt"] = max(piece["S"].get("rt", 0.0), 5.0)

    piece["qaoa"] = qaoa
    piece["arrangement_profile"] = f
    return piece

def compose(p, progress=lambda frac,msg="":None):
    """Quantum composition followed by QAOA-selected genre arrangement."""
    piece = _QMC_BASE_COMPOSE(p, progress)
    piece["S"] = dict(piece["S"])  # avoid mutating global sound presets between compositions
    piece["S"]["rain_enabled"] = piece["genre"] == "Lo-fi hip-hop"
    piece["S"]["rain_level"] = float(p.get("rain_level", 0.035))
    qaoa = qaoa_arrangement(piece["genre"])
    piece = _apply_genre_arrangement(piece, qaoa)
    return piece

# ---------------- MIDI: real GM instrument program assignment + track-specific swing ----------------
def midi_bytes(piece, filename="quantum_music.mid"):
    mid = MidiFile(type=1, ticks_per_beat=480)
    bpm = piece["p"]["bpm"]
    ts = piece["ts"]
    genre = piece["genre"]
    slots = piece["slots"]
    swing = float(piece.get("style",{}).get("swing",0.5))

    # GM program numbers are zero-based. Each track has its own instrument identity.
    programs = {
        "melody": 65 if genre=="Jazz (swing)" else (4 if genre=="Lo-fi hip-hop" else (81 if genre=="EDM / folk-house" else 0)),
        "harmony": 0,
        "pad": 0 if genre=="Jazz (swing)" else (4 if genre=="Lo-fi hip-hop" else (89 if genre=="EDM / folk-house" else 89)),
        "bass": 32 if genre=="Jazz (swing)" else (38 if genre=="EDM / folk-house" else 33),
        "arp": 81 if genre=="EDM / folk-house" else (4 if genre=="Lo-fi hip-hop" else 10),
        "guitar": 65 if genre=="Jazz (swing)" else (24 if genre=="Pop / cinematic (default)" else 25),
        "hook": 4 if genre=="Lo-fi hip-hop" else (81 if genre=="EDM / folk-house" else 0),
    }
    tracks = [
        ("melody",piece["melody"],0,programs["melody"]),
        ("harmony",piece["harmony"],4,programs["harmony"]),
        ("pad",piece["pad"],1,programs["pad"]),
        ("bass",piece["bass"],2,programs["bass"]),
        ("arp",piece["arp"],3,programs["arp"]),
        ("guitar",piece.get("guitar",[]),5,programs["guitar"]),
        ("hook",piece.get("hook",[]),6,programs["hook"]),
        ("drums",piece.get("drums",[]),9,None),
    ]

    def tick(slot, track_name, pitch):
        base = int(round(slot*240))
        odd = int(slot)%2==1
        if not odd: return base
        # Swing is applied only to selected subdivisions/tracks, never to every instrument.
        if genre == "Jazz (swing)" and (
            track_name in ("melody","guitar") or (track_name=="drums" and pitch in (51,42))
        ):
            return base + int(round((swing-0.5)*480))
        if genre == "Lo-fi hip-hop" and (
            track_name in ("melody","arp","hook") or (track_name=="drums" and pitch==42)
        ):
            return base + int(round((swing-0.5)*480))
        return base

    for track_name, notes, channel, program in tracks:
        tr = MidiTrack()
        mid.tracks.append(tr)
        tr.append(MetaMessage("track_name",name=track_name,time=0))
        if track_name=="melody":
            tr.append(MetaMessage("set_tempo",tempo=bpm2tempo(bpm),time=0))
            tr.append(MetaMessage("time_signature",numerator=ts["num"],denominator=ts["den"],time=0))
        if program is not None:
            tr.append(Message("program_change",channel=channel,program=int(program),time=0))
        events=[]
        for s,d,pitch,vel in notes:
            start=tick(s,track_name,pitch)
            end=max(start+1,tick(s+d,track_name,pitch)-5)
            events.append((start,1,Message("note_on",channel=channel,note=int(pitch),velocity=int(vel))))
            events.append((end,0,Message("note_off",channel=channel,note=int(pitch),velocity=0)))
        # Sidechain-style pad ducking via channel volume automation on the pad channel only.
        if genre=="EDM / folk-house" and track_name=="pad" and piece.get("S",{}).get("sidechain",True):
            for s,d,pitch,vel in piece.get("drums",[]):
                if pitch==36:
                    t=tick(s,"drums",pitch)
                    events.append((t,0,Message("control_change",channel=channel,control=7,value=58)))
                    events.append((t+75,1,Message("control_change",channel=channel,control=7,value=100)))
        events.sort(key=lambda e:(e[0],e[1]))
        last=0
        for tick_pos,_,msg in events:
            msg.time=max(0,tick_pos-last)
            tr.append(msg)
            last=tick_pos
    mid.save(filename)
    with open(filename,"rb") as f:
        return f.read()

# ---------------- SoundFont discovery + renderer ----------------
def find_soundfont():
    candidates = []
    for root in ("/usr/share/sounds/sf2","/usr/share/soundfonts","/usr/local/share/sounds/sf2"):
        candidates += glob.glob(os.path.join(root,"*.sf2"))
    preferred = [p for p in candidates if "FluidR3_GM" in os.path.basename(p)]
    return preferred[0] if preferred else (candidates[0] if candidates else None)

SOUNDFONT_PATH = find_soundfont()
FLUIDSYNTH_PATH = shutil.which("fluidsynth")
print("SoundFont:", SOUNDFONT_PATH or "not found — FM fallback")
print("FluidSynth:", FLUIDSYNTH_PATH or "not found — FM fallback")

def _postprocess_audio(audio, piece):
    audio = np.asarray(audio,dtype=np.float32)
    if audio.ndim==2:
        audio = audio.mean(axis=1)
    if piece["genre"]=="Lo-fi hip-hop":
        # Warm low-pass, subtle vinyl crackle, slow wow/flutter and a CONTINUOUS rain/hiss bed.
        cutoff = 5200
        audio = sosfilt(butter(2,cutoff,fs=SR,output="sos"),audio)
        audio = add_vinyl_texture(audio,piece["S"].get("vinyl",0.012),seed=31)
        t=np.arange(len(audio))/SR
        audio *= (1+piece["S"].get("wow",0.0015)*np.sin(2*np.pi*0.23*t))
        if piece["S"].get("rain_enabled", True):
            rng=np.random.default_rng(2026)
            rain=rng.standard_normal(len(audio)).astype(np.float32)
            rain=sosfilt(butter(3,[420,8200],btype="bandpass",fs=SR,output="sos"),rain)
            rain=rain/(np.sqrt(np.mean(rain**2))+1e-9)
            # Tiny slow level drift makes it resemble steady rainfall rather than a static test tone.
            drift=0.92+0.08*np.sin(2*np.pi*0.035*t+0.7)
            rain_level=float(piece["S"].get("rain_level",0.035))
            audio=audio + rain_level*rain*drift
    peak=np.max(np.abs(audio)) if len(audio) else 0
    return audio/(peak+1e-9)*0.92 if peak else audio

def render_audio(piece, renderer="soundfont"):
    """
    Preferred: FluidSynth renders MIDI through sampled GM instruments.
    Fallback: original FM engine, with global swing disabled to avoid swinging every track.
    """
    global SOUNDFONT_PATH, FLUIDSYNTH_PATH
    if renderer=="soundfont" and SOUNDFONT_PATH and FLUIDSYNTH_PATH:
        with tempfile.TemporaryDirectory() as td:
            midi_path=os.path.join(td,"piece.mid")
            wav_path=os.path.join(td,"piece.wav")
            midi_bytes(piece,midi_path)
            cmd=[FLUIDSYNTH_PATH,"-ni","-F",wav_path,"-r",str(SR),SOUNDFONT_PATH,midi_path]
            proc=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
            if proc.returncode==0 and os.path.exists(wav_path):
                rate,audio=wavfile.read(wav_path)
                if rate!=SR:
                    raise RuntimeError(f"FluidSynth rendered {rate} Hz; expected {SR} Hz")
                if audio.dtype.kind in "iu":
                    audio=audio.astype(np.float32)/np.iinfo(audio.dtype).max
                return _postprocess_audio(audio,piece)
            print("FluidSynth rendering failed; using FM fallback:",proc.stderr[-500:])
    # FM fallback: preserve straight timing globally rather than applying one swing value to all tracks.
    old_swing=piece["style"].get("swing",0.5)
    piece["style"]["swing"]=0.5
    try:
        audio=_QMC_BASE_RENDER_AUDIO(piece)
    finally:
        piece["style"]["swing"]=old_swing
    return _postprocess_audio(audio,piece)

# ---------------- Evaluation: structure + rendered audio descriptors ----------------
def evaluate(piece, audio=None):
    ev=_QMC_BASE_EVALUATE(piece)
    genre=piece["genre"]
    drums=[e[2] for e in piece.get("drums",[])]
    if genre=="Jazz (swing)":
        checks={
            "Walking bass line": len(piece["bass"])>=piece["bars"]*4,
            "Piano comping": len(piece["pad"])>=piece["bars"]*2,
            "Ride cymbal": 51 in drums,
            "Seventh-chord vocabulary": piece["p"].get("sevenths",False),
            "Selective swing (not global)": piece["style"]["swing"]>=0.60,
            "Call-and-response layer": len(piece.get("guitar",[]))>0,
        }
    elif genre=="EDM / folk-house":
        kick_slots=[e[0]%piece["slots"] for e in piece.get("drums",[]) if e[2]==36]
        checks={
            "Four-on-the-floor kick": all(s in kick_slots for s in (0,2,4,6)),
            "Electronic bass pulse": len(piece["bass"])>=piece["bars"]*4,
            "Build/drop drum variation": len([e for e in piece["drums"] if e[2]==38])>=piece["bars"],
            "Synth arp/lead layer": len(piece["arp"])>0,
            "Acoustic guitar removed": len(piece.get("guitar",[]))==0,
        }
    elif genre=="Lo-fi hip-hop":
        checks={
            "Electric-piano timbre assignment": piece["S"].get("lead")=="epiano",
            "Sparse rhythm setting": piece["p"]["busy"]=="Sparse",
            "Low-level vinyl texture": piece["S"].get("vinyl",0)>0,
            "Continuous rain ambience enabled": piece["S"].get("rain_enabled",False),
            "Soft drum velocities": all(e[3]<=65 for e in piece.get("drums",[])),
            "Repeated motif layer": len(piece.get("hook",[]))>0,
        }
    else:
        checks={
            "Pop backbeat": 38 in drums,
            "Moderate tempo": 90<=piece["p"]["bpm"]<=125,
            "Hook repetition": len(piece.get("hook",[]))>0,
            "Guitar/piano support": len(piece.get("guitar",[]))+len(piece.get("pad",[]))>0,
        }
    ev["genre_checks"]=checks
    ev["genre_score"]=100*sum(bool(v) for v in checks.values())/max(1,len(checks))
    ev["qaoa_cost"]=piece["qaoa"]["expected_cost"]
    ev["qaoa_classical_optimum"]=piece["qaoa"]["classical_optimum"]
    ev["qaoa_features"]=piece["arrangement_profile"]
    if audio is not None and len(audio):
        x=np.asarray(audio,dtype=float)
        if x.ndim==2: x=x.mean(axis=1)
        rms=float(np.sqrt(np.mean(x*x)))
        freqs,power=welch(x,fs=SR,nperseg=min(4096,len(x)))
        centroid=float(np.sum(freqs*power)/(np.sum(power)+1e-12))
        low_ratio=float(np.sum(power[freqs<180])/(np.sum(power)+1e-12))
        ev["audio_features"]={"rms":rms,"spectral_centroid_hz":centroid,"low_frequency_ratio":low_ratio}
    else:
        ev["audio_features"]={}
    return ev

# ---------------- Enhanced dashboard ----------------

def _make_markov_stateprep_circuit(piece):
    """Representative QuPoly state-preparation circuit for the strong-beat Markov distribution."""
    probs = np.asarray(piece["demo"]["probs"], dtype=float)
    n = int(np.log2(len(probs)))
    qc = QuantumCircuit(n, n)
    qc.append(StatePreparation(np.sqrt(probs)), range(n))
    qc.barrier()
    qc.measure(range(n), range(n))
    return qc

def _plot_probability_profile(piece, grover_data):
    probs = np.asarray(piece["demo"]["probs"], dtype=float)
    counts = np.asarray(grover_data["counts"], dtype=float)
    x = np.arange(len(piece["MEL"]))
    labels = [NOTE_NAMES[m % 12] + str(m // 12 - 1) for m in piece["MEL"]]
    fig, ax = plt.subplots(figsize=(11, 3.8))
    width = 0.38
    ax.bar(x-width/2, probs[:len(x)], width, label="Before Grover (Markov)")
    ax.bar(x+width/2, counts[:len(x)], width, label="After Grover (measured)")
    for idx in piece["demo"]["marked"]:
        ax.axvspan(idx-0.5, idx+0.5, color="gold", alpha=0.13)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35)
    ax.set_ylabel("Probability")
    ax.set_title("Grover amplitude amplification: chord-tone probability")
    ax.legend(); plt.tight_layout()
    return fig

def _plot_rhythm_grid(piece):
    arr=np.asarray(piece["rhythm"],dtype=int)
    fig,ax=plt.subplots(figsize=(10,3.2))
    ax.imshow(arr,aspect="auto",interpolation="nearest",cmap="Blues",vmin=0,vmax=1)
    ax.set_xlabel("Eighth-note slot"); ax.set_ylabel("Bar")
    ax.set_title("PQCA-generated rhythm grid (1 = onset)")
    ax.set_xticks(range(arr.shape[1]))
    plt.tight_layout()
    return fig

def _plot_audio_spectrum(audio):
    x=np.asarray(audio,dtype=float)
    if x.ndim==2: x=x.mean(axis=1)
    freqs,power=welch(x,fs=SR,nperseg=min(4096,len(x)))
    fig,ax=plt.subplots(figsize=(10,3))
    ax.semilogx(np.maximum(freqs,1),10*np.log10(power+1e-12))
    ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Power (dB)")
    ax.set_title("Rendered audio power spectrum")
    ax.grid(True,alpha=0.2); plt.tight_layout()
    return fig

def make_dashboard_v2():
    genres=["Pop / cinematic (default)","Jazz (swing)","EDM / folk-house","Lo-fi hip-hop","Ambient","Country","Rock"]
    style= W.Dropdown(options=genres,value=genres[0],description="Genre")
    key= W.Dropdown(options=[(n,i) for i,n in enumerate(NOTE_NAMES)],value=0,description="Key")
    bpm= W.IntSlider(value=108,min=45,max=170,description="Tempo",continuous_update=False)
    seconds= W.IntSlider(value=30,min=20,max=90,step=5,description="Length (s)",continuous_update=False)
    seed= W.IntText(value=7,description="Seed")
    use_random= W.Checkbox(value=False,description="Randomize each composition")
    renderer= W.Dropdown(options=[("SoundFont samples (recommended)","soundfont"),("Original FM fallback","fm")],value="soundfont",description="Renderer")
    rain_level= W.FloatSlider(value=0.035,min=0.0,max=0.08,step=0.005,description="Lo-fi rain level",continuous_update=False)
    show_circuits= W.Checkbox(value=True,description="Show all quantum circuits")
    compose_btn= W.Button(description="🎼 Compose",button_style="primary",layout=W.Layout(width="170px",height="40px"))
    status= W.HTML()
    tabs_out=[W.Output() for _ in range(5)]
    tabs=W.Tab(children=tabs_out)
    for i,t in enumerate(["🎧 Listen","🎹 Arrangement","⚛️ All circuits","📈 Diagnostics","📊 Validation"]): tabs.set_title(i,t)

    def apply_defaults(change=None):
        st=STYLES[style.value]
        bpm.value=st["bpm"]
    style.observe(apply_defaults,names="value")

    def on_compose(_=None):
        compose_btn.disabled=True
        for o in tabs_out: o.clear_output()
        try:
            st=STYLES[style.value]
            p={
                "genre":style.value,"key":key.value,"scale":st["scale"],"bpm":bpm.value,
                "notes":st["notes"],"busy":st["busy"],"seconds":seconds.value,"ts":st["ts"],
                "sevenths":st["sevenths"],"swing":st["swing"],"random_seed":use_random.value,
                "seed":seed.value,"sound":st["sound"],"show_circuits":True,"show_expl":True,
                "rain_level":rain_level.value
            }
            def cb(frac,msg=""):
                status.value=f"<span>{frac*100:.0f}% — {msg}</span>"
            piece=compose(p,cb)
            audio=render_audio(piece,renderer=renderer.value)
            ev=evaluate(piece,audio)
            wb=wav_bytes(audio)
            mb=midi_bytes(piece,"quantum_music.mid")
            with tabs_out[0]:
                display(HTML(
                    f"<h3>{piece['genre']}</h3><p>{NOTE_NAMES[p['key']]} · {p['bpm']} BPM · "
                    f"{piece['bars']} bars · {piece['total']:.1f} seconds</p>"
                ))
                display(Audio(audio,rate=SR))
                display(HTML("<p><a download='quantum_music.wav' href='data:audio/wav;base64,"+
                             base64.b64encode(wb).decode()+"'>⬇ Download WAV</a> &nbsp; | &nbsp; "+
                             "<a download='quantum_music.mid' href='data:audio/midi;base64,"+
                             base64.b64encode(mb).decode()+"'>⬇ Download MIDI</a></p>"))
                renderer_name=("FluidSynth + SoundFont samples" if renderer.value=="soundfont" and SOUNDFONT_PATH and FLUIDSYNTH_PATH else "FM synthesis fallback")
                display(HTML("<p><b>Renderer:</b> "+renderer_name+"</p>"))
                if piece["genre"]=="Lo-fi hip-hop":
                    display(HTML(f"<p><b>Continuous rain bed:</b> enabled · level {piece['S']['rain_level']:.3f} (audio layer; not exported as MIDI)</p>"))

            with tabs_out[1]:
                fig=fig_piano_roll(piece); display(fig); plt.close(fig)
                display(HTML("<h4>Instrument roles</h4><ul>"+
                    "<li>Melody: genre-selected lead instrument</li>"+
                    "<li>Bass: genre-specific foundation</li>"+
                    "<li>Pad/harmony: sustained chords or comping</li>"+
                    "<li>Drums: genre-specific percussion (none for Ambient)</li>"+
                    "<li>Hook/response: repeated motif or call-and-response layer</li></ul>"))

            # Generate one representative diagnostic sample for the same circuit families.
            gd=grover_demo_data(piece,shots=2048)
            markov_qc=_make_markov_stateprep_circuit(piece)
            seed_state=[1 if i in (0,3,5) else 0 for i in range(piece["slots"])]
            pqca_qc=pqca_circuit(seed_state)
            qrng_qc=qrng_circuit(2)
            bell_qc=bell_circuit()
            circuit_items=[
                ("1 · Hadamard QRNG — selects the harmonic progression",qrng_qc),
                ("2 · PQCA — rhythm evolution circuit (initial bar)",pqca_qc),
                ("3 · Quantum Markov / QuPoly — state preparation and measurement",markov_qc),
                ("4 · Grover — state preparation + chord-tone oracle + diffusion",gd["qc"]),
                ("5 · Bell pair — H + CNOT entanglement for accompaniment correlation",bell_qc),
                ("6 · QAOA — genre-conditioned arrangement decisions",piece["qaoa"]["qc"]),
            ]
            with tabs_out[2]:
                if show_circuits.value:
                    for caption,qc in circuit_items:
                        display(HTML(f"<h4>{caption}</h4>"))
                        try:
                            fig=qc.draw("mpl",fold=-1); display(fig); plt.close(fig)
                        except Exception:
                            print(qc.draw("text"))
                else:
                    print("Enable 'Show all quantum circuits' and compose again.")
                q=piece["qaoa"]
                display(HTML(
                    f"<h4>QAOA result</h4><p>Method: {q['method']} · selected state: "
                    f"{q['selected_state']:03b} · expected cost: {q['expected_cost']:.3f} · "
                    f"classical optimum: {q['classical_optimum']:.3f}</p>"
                ))

            with tabs_out[3]:
                display(HTML("<h4>Grover probability amplification</h4>"))
                fig=_plot_probability_profile(piece,gd); display(fig); plt.close(fig)
                fig=fig_grover(piece,gd); display(fig); plt.close(fig)
                fig=_plot_rhythm_grid(piece); display(fig); plt.close(fig)
                fig=fig_eval(ev); display(fig); plt.close(fig)
                fig=_plot_audio_spectrum(audio); display(fig); plt.close(fig)
                fig=fig_melody_identity(piece); display(fig); plt.close(fig)
                q=piece["qaoa"]
                states=np.arange(8)
                counts=np.array([q["counts"].get(int(i),0) for i in states])
                fig,ax=plt.subplots(figsize=(8,2.8))
                ax.bar([f"{i:03b}" for i in states],counts)
                ax.set_xlabel("Arrangement state (q2 q1 q0)"); ax.set_ylabel("Measurement counts")
                ax.set_title("QAOA measurement histogram")
                plt.tight_layout(); display(fig); plt.close(fig)

            with tabs_out[4]:
                checks="".join(f"<li>{'✓' if ok else '✗'} {label}</li>" for label,ok in ev["genre_checks"].items())
                display(HTML(f"<h4>Structural genre alignment: {ev['genre_score']:.0f}%</h4><ul>{checks}</ul>"))
                mm=ev.get("melody_metrics", {})
                display(HTML(
                    f"<p><b>Melody strategy:</b> {ev.get('melody_strategy','genre-conditioned')}</p>"
                    f"<p>Notes: {mm.get('note_count',0)} · Mean interval: {mm.get('mean_abs_interval_semitones',0):.1f} semitones · "
                    f"Repeated-note ratio: {mm.get('repeated_note_ratio',0)*100:.1f}% · "
                    f"Stepwise ratio: {mm.get('stepwise_ratio',0)*100:.1f}% · "
                    f"Pitch range: {mm.get('pitch_range_semitones',0)} semitones</p>"
                    f"<p>In-scale notes: {ev['in_scale']} · Ends on tonic: {ev['ends_on_tonic']} · "
                    f"Strong-beat chord tones: {ev['ct_ours']*100:.1f}% · Offbeat onset ratio: {ev['offbeat_ratio']*100:.1f}%</p>"
                ))
                if ev.get("audio_features"):
                    af=ev["audio_features"]
                    display(HTML(f"<h4>Rendered audio descriptors</h4><ul><li>RMS energy: {af['rms']:.4f}</li>"+
                                 f"<li>Spectral centroid: {af['spectral_centroid_hz']:.0f} Hz</li>"+
                                 f"<li>Low-frequency energy ratio (&lt;180 Hz): {af['low_frequency_ratio']:.3f}</li></ul>"))
                display(HTML("<p><b>Interpretation:</b> These are transparent structural/audio indicators, not a trained genre classifier or proof of subjective genre identity.</p>"))
            status.value="✅ Composition complete."
        except Exception:
            status.value="<b style='color:#b00'>Composition failed. See error below.</b>"
            with tabs_out[0]:
                print(traceback.format_exc())
        finally:
            compose_btn.disabled=False

    compose_btn.on_click(on_compose)
    ui=W.VBox([
        W.HTML("<h2>🎼 Quantum Genre Music Composer</h2><p>Hybrid quantum composition + Grover harmony search + QAOA arrangement + SoundFont rendering</p>"),
        W.HBox([style,key,bpm,seconds],layout=W.Layout(flex_flow="row wrap")),
        W.HBox([seed,use_random,renderer,rain_level,show_circuits],layout=W.Layout(flex_flow="row wrap")),
        compose_btn,status,tabs
    ])
    return ui,on_compose

ui_v2, compose_v2_click = make_dashboard_v2()
display(ui_v2)


SoundFont: /usr/share/sounds/sf2/FluidR3_GM.sf2
FluidSynth: /usr/bin/fluidsynth


## 12. Genre-specific melodic identity (quantum-sampled)
This final extension replaces the generic melody prior with genre-conditioned melodic behaviour. Genre priors shape the amplitudes passed to quantum state preparation; Qiskit measurement still selects each note, and Grover amplification remains in the strong-beat path. The existing QRNG, PQCA, Bell-pair, QAOA, arrangement, renderer and MIDI pipeline are retained.


In [ ]:
# ============================================================
# 12. GENRE-SPECIFIC MELODIC IDENTITY (QUANTUM-SAMPLED)
# ============================================================
# Runs after the complete QRNG + PQCA + Markov/Grover + Bell + QAOA + SoundFont pipeline.
# It replaces the generic melody probability prior, not the quantum sampling stage.

if "_QMC_PRE_MELODY_COMPOSE" not in globals():          # capture once (safe to re-run)
    _QMC_PRE_MELODY_COMPOSE = compose
    _QMC_PRE_MELODY_EVALUATE = evaluate


def _genre_melody_distribution(MEL, nq, prev_i, prev_dir, chord_tones, strong,
                               genre, slot, note_no, motif):
    """Genre-specific musical prior; returned probabilities are encoded in a quantum state."""
    n = len(MEL)
    w = np.zeros(2**nq, dtype=float)
    chord_pcs = {int(x) % 12 for x in chord_tones}
    chord_ids = [i for i, pitch in enumerate(MEL) if pitch % 12 in chord_pcs]
    centre = (n - 1) / 2
    sigma = max(1.8, n * 0.38)

    # Shared baseline: local continuity, modest leap recovery, and less central gravity.
    for i, pitch in enumerate(MEL):
        d = abs(i - prev_i)
        base = {0: .32, 1: 2.2, 2: 1.7, 3: .95, 4: .62, 5: .38}.get(d, .22)
        register = math.exp(-((i-centre)**2)/(2*sigma**2))
        w[i] = base * (.60 + .40*register)
        if abs(prev_dir) >= 3 and d in (1, 2) and np.sign(i-prev_i) == -np.sign(prev_dir):
            w[i] *= 1.8
        if i in chord_ids:
            w[i] *= 1.45 if strong else 1.10
        elif strong:
            w[i] *= .72

    # Correct dashboard labels are used here (not the stale labels in the first prototype).
    if genre == "Pop / cinematic (default)":
        for i in range(n):
            d = abs(i-prev_i)
            if d <= 2: w[i] *= 1.35
            if d >= 4: w[i] *= .62
        if len(motif) >= 4:
            target = motif[note_no % len(motif)]
            for i in range(n):
                d = abs(i-target)
                if d == 0: w[i] *= 3.0
                elif d == 1: w[i] *= 1.45
        if slot >= 6:
            for i in chord_ids: w[i] *= 1.4

    elif genre == "Jazz (swing)":
        for i in range(n):
            d = abs(i-prev_i)
            if d in (2, 3, 4): w[i] *= 1.75
            elif d >= 5: w[i] *= 1.2
            elif d == 0: w[i] *= .48
            if i not in chord_ids and not strong: w[i] *= 1.25
        if slot % 2: w *= 1.10
        if note_no % 8 in (6, 7):
            for i in chord_ids: w[i] *= 1.45

    elif genre == "EDM / folk-house":
        if len(motif) >= 4:
            target = motif[note_no % min(4, len(motif))]
            for i in range(n):
                d = abs(i-target)
                if d == 0: w[i] *= 3.8
                elif d == 1: w[i] *= 1.35
        for i in range(n):
            d = abs(i-prev_i)
            if d in (0, 2, 4): w[i] *= 1.35
            elif d == 1: w[i] *= .78
        if slot in (0, 4):
            for i in chord_ids: w[i] *= 1.7

    elif genre == "Lo-fi hip-hop":
        for i in range(n):
            d = abs(i-prev_i)
            if d == 0: w[i] *= 1.7
            elif d <= 2: w[i] *= 1.4
            elif d >= 4: w[i] *= .42
        if len(motif) >= 4:
            target = motif[note_no % min(4, len(motif))]
            for i in range(n):
                d = abs(i-target)
                if d == 0: w[i] *= 2.7
                elif d == 1: w[i] *= 1.5
        if slot in (0, 4):
            for i in chord_ids: w[i] *= 1.3

    elif genre == "Ambient":
        for i in range(n):
            d = abs(i-prev_i)
            if d <= 2: w[i] *= 1.8
            elif d in (3, 4): w[i] *= .9
            else: w[i] *= .72
        if slot == 0:
            for i in chord_ids: w[i] *= 1.65

    elif genre == "Country":
        for i in range(n):
            d = abs(i-prev_i)
            if d <= 2: w[i] *= 1.5
            elif d in (3, 4): w[i] *= 1.2
            else: w[i] *= .55
        if len(motif) >= 4:
            target = motif[note_no % min(4, len(motif))]
            for i in range(n):
                if i == target: w[i] *= 1.7
        if slot >= 6:
            for i in chord_ids: w[i] *= 1.5

    elif genre == "Rock":
        for i in range(n):
            d = abs(i-prev_i)
            if d == 0: w[i] *= 1.5
            elif d in (2, 3, 4): w[i] *= 1.45
            elif d >= 5: w[i] *= 1.15

    w[n:] = 0.0
    total = float(w.sum())
    if not np.isfinite(total) or total <= 0:
        w[:n] = 1.0/n
    else:
        w /= total
    return w


def _sample_genre_note(MEL, nq, prev_i, prev_dir, tones, strong, genre,
                       slot, note_no, motif, force=None):
    """Encode genre prior in amplitudes, optionally apply Grover, then measure."""
    probs = _genre_melody_distribution(
        MEL, nq, prev_i, prev_dir, tones, strong, genre, slot, note_no, motif
    )
    marked = [i for i, pitch in enumerate(MEL) if pitch % 12 in set(tones)]
    if force is not None:
        probs = np.zeros(2**nq, dtype=float)
        probs[int(force)] = 1.0
        qc, k, p_before = amplitude_amplification_circuit(probs, marked, k=0)
    else:
        adjusted = strong_prior(probs, marked) if (strong and 0 < len(marked) < len(MEL)) else None
        if adjusted is not None:
            probs = adjusted
            qc, k, p_before = amplitude_amplification_circuit(probs, marked)
        else:
            qc, k, p_before = amplitude_amplification_circuit(probs, marked, k=0)
    idx = int(run_circuit(qc)[0], 2)
    if idx >= len(MEL): idx = int(np.argmax(probs[:len(MEL)]))
    return idx, qc, k, p_before, probs


def _melody_metrics(piece):
    notes = piece.get("melody", [])
    pitches = [int(e[2]) for e in notes]
    intervals = np.diff(pitches) if len(pitches) > 1 else np.array([], dtype=int)
    mel = piece.get("MEL", [])
    indices = [mel.index(m) for m in pitches if m in mel]
    steps = np.diff(indices) if len(indices) > 1 else np.array([], dtype=int)
    return {
        "note_count": len(pitches),
        "mean_abs_interval_semitones": float(np.mean(np.abs(intervals))) if len(intervals) else 0.0,
        "max_interval_semitones": int(np.max(np.abs(intervals))) if len(intervals) else 0,
        "repeated_note_ratio": float(np.mean(steps == 0)) if len(steps) else 0.0,
        "stepwise_ratio": float(np.mean(np.abs(steps) <= 2)) if len(steps) else 0.0,
        "pitch_range_semitones": int(max(pitches)-min(pitches)) if pitches else 0,
    }


def compose(p, progress=lambda frac, msg="": None):
    """Existing full pipeline plus genre-specific quantum-sampled melodic phrasing."""
    # Skip the old generic melody pass inside the legacy composer: all its other
    # outputs (QRNG progression, PQCA rhythm, Bell pairs, bass/pads/drums, QAOA)
    # are retained, and the genre-specific quantum melody is sampled below once.
    global quantum_note
    _qmc_original_quantum_note = quantum_note
    def _qmc_skip_legacy_note(mel, nq, prev_i, prev_dir, tones, strong, force=None, genre="Pop / cinematic (default)"):
        idx = int(force) if force is not None else int(np.clip(prev_i, 0, len(mel)-1))
        probs = np.zeros(2**nq, dtype=float); probs[idx] = 1.0
        return idx, None, 0, 0.0, probs
    quantum_note = _qmc_skip_legacy_note
    try:
        piece = _QMC_PRE_MELODY_COMPOSE(p, progress)
    finally:
        quantum_note = _qmc_original_quantum_note
    genre, MEL, nq = piece["genre"], piece["MEL"], piece["NQ"]
    melody, log, demo = [], [], None
    prev_i, prev_dir = len(MEL)//2, 0
    motif, note_no = [], 0

    for bar in range(piece["bars"]):
        degree = piece["degs"][bar]
        tones = piece["CH"][degree]["tones"]
        last_bar = (bar == piece["bars"]-1)
        onsets = [0] if last_bar else [s for s in range(piece["slots"]) if piece["rhythm"][bar][s]]
        for j, slot in enumerate(onsets):
            strong = slot in piece["ts"]["strong"]
            force = piece["final_i"] if last_bar else None
            idx, qc, k, p_before, used = _sample_genre_note(
                MEL, nq, prev_i, prev_dir, tones, strong, genre, slot, note_no, motif, force
            )
            # The motif is learned from measured notes in this generated piece.
            if len(motif) < 4 and not last_bar:
                motif.append(idx)
            if demo is None and strong and force is None and k > 0:
                demo = {"probs": used,
                        "marked": [i for i,m in enumerate(MEL) if m % 12 in set(tones)],
                        "label": piece["labels"][bar], "qc": qc}
            log.append((bar, slot, strong, degree, k, p_before, idx))
            end = onsets[j+1] if j+1 < len(onsets) else piece["slots"]
            melody.append((bar*piece["slots"]+slot, end-slot, int(MEL[idx]), 84 if strong else 72))
            prev_dir, prev_i = idx-prev_i, idx
            note_no += 1

    piece["melody"], piece["log"], piece["demo"] = melody, log, demo
    piece["melody_metrics"] = _melody_metrics(piece)
    piece["melody_strategy"] = {
        "Pop / cinematic (default)": "learned hook motif, stepwise contour and phrase returns",
        "Jazz (swing)": "intervallic improvisation, weak-beat colour tones and chord-aware phrase endings",
        "EDM / folk-house": "repeating synth ostinato and chord-anchored rhythmic motif",
        "Lo-fi hip-hop": "relaxed repeated motif, close intervals and restrained note density",
        "Ambient": "spacious sustained contour without a forced repeating hook",
        "Country": "major-pentatonic storytelling contour with phrase-end chord tones",
        "Rock": "riff-oriented repeated notes and wider interval movement",
    }.get(genre, "genre-conditioned contour")

    # Regenerate hook from the new quantum-generated melody, rather than retaining
    # the hook made from the previous generic melody.
    if genre == "Pop / cinematic (default)" and piece.get("arrangement_profile", {}).get("hook_motif", True):
        piece["hook"] = _add_pop_hook(piece)
    elif genre == "Lo-fi hip-hop" and piece.get("arrangement_profile", {}).get("loop_motif", True):
        piece["hook"] = _add_pop_hook(piece)
    return piece


def evaluate(piece, audio=None):
    ev = _QMC_PRE_MELODY_EVALUATE(piece, audio)
    metrics = piece.get("melody_metrics", _melody_metrics(piece))
    ev["melody_metrics"] = metrics
    ev["melody_strategy"] = piece.get("melody_strategy", "genre-conditioned")
    ev["genre_checks"]["At least three distinct melody pitches"] = len(set(e[2] for e in piece.get("melody", []))) >= 3
    ev["genre_checks"]["Melodic interval variation"] = metrics["max_interval_semitones"] >= 3
    ev["genre_score"] = 100*sum(bool(v) for v in ev["genre_checks"].values())/max(1,len(ev["genre_checks"]))
    return ev


def fig_melody_identity(piece):
    notes = piece.get("melody", [])
    x = [e[0] for e in notes]
    y = [e[2] for e in notes]
    fig, ax = plt.subplots(figsize=(11, 3.2))
    if x: ax.plot(x, y, marker="o", linewidth=1.4)
    ax.set_xlabel("Eighth-note slot")
    ax.set_ylabel("MIDI pitch")
    ax.set_title("Genre-conditioned melody contour — " + piece["genre"])
    ax.grid(True, alpha=.22)
    plt.tight_layout()
    return fig

print("Genre-specific quantum melody engine loaded.")


Genre-specific quantum melody engine loaded.


## 13. Rich phrase engine – riffs, arpeggios, counter-melody, fast rhythm, murki / harkat / meend

The sections above produce a *skeleton*: chords, PQCA rhythm, one quantum-sampled melody line. This section **develops** that skeleton so it no longer sounds like one plain line over block chords. It runs after everything else, so nothing above has to change.

| What was missing | What is added | Quantum idea used |
|---|---|---|
| Arpeggios too simple (`0-1-2-1` loop) | Arpeggio contour is a **walk on a ring of chord tones** (up-biased, with momentum, folded into a zig-zag ladder) | **Discrete-time quantum walk** (coin qubit + controlled ±1 shift, position measured every step) |
| No riffs | **PQCA proposes 8 candidate riff rhythms**, Grover picks one of the 2 best; pitches are sampled once relative to the chord root so the riff transposes with the chords. Rock gets palm-muted chugs + power-chord accents + root-locked bass | **PQCA + Grover** (P(best 2) 25 % → ≈100 % with k = 1) |
| No counter-melody | A second line in the lower register: contrary motion, 3rds/6ths against the lead on strong beats. In *Bell bars* it becomes an **echo (canon) a third below** | **Grover consonance oracle** + **Bell-pair bit** shared by melody and counter-melody |
| No fast rhythmic changes | Each bar picks a mode: 16th-note run, eighth-triplet burst, push/syncopation, stop-time; drum fills, crashes, 16th hats, ghost notes | QuPoly state-prep chooses the mode; QRNG decides every yes/no |
| No *harkat / murki* | **murki** (quick neighbour turn), **harkat** (twisting run into the note), **khatka**, **kan** (grace note) and **meend** (true continuous glide with MIDI pitch-bend). Also Western turns/trills and blues bends | The murki/harkat contour is a **short quantum walk around the target note** |
| Phrases never returned | The hook is now a real **motif return**, repeated every 4 bars as a *melodic sequence* that follows the chord, replacing (not doubling) the plain melody | – |

**Controls:** after running this cell a small panel appears under the dashboard. Set *Riffs / Arpeggios / Counter-melody / Ornaments / Rhythmic variety* to Auto, Off, Light or Full, choose an *Ornament flavour* (try **Hindustani**) and a *Raga* (try **Bhairav**), then press **🎼 Compose** in the dashboard above.

**Honest limits:** simulator only, no quantum speed-up. Classical music theory still defines the allowed notes; the circuits make the *choices*. Run the notebook top to bottom once (*Runtime → Run all*).


In [ ]:
# ============================================================
# 13. RICH PHRASE ENGINE  (riffs · arpeggios · counter-melody · fast rhythm · ornaments)
# ============================================================
# Runs AFTER the whole existing pipeline. The old pipeline still produces the "skeleton"
# (chords, PQCA rhythm, QuPoly/Grover melody, Bell bits, QAOA arrangement). This cell then
# develops that skeleton with new quantum sub-circuits:
#   * quantum WALK            -> arpeggio contours AND murki / harkat / khatka ornaments
#   * Grover over PQCA riffs  -> picks the riff (rock / pop / EDM / country)
#   * Grover consonance oracle-> counter-melody (3rds / 6ths against the lead)
#   * Bell-pair bit           -> counter-melody is an ENTANGLED echo (canon) or free counterpoint
#   * QuPoly state-prep       -> rhythm-mode choice per bar, ornament type, riff pitches
#   * Hadamard QRNG           -> every "should this note be ornamented?" yes/no decision
import bisect

if "_QMC_PRE_RICH_COMPOSE" not in globals():
    _QMC_PRE_RICH_COMPOSE  = compose
    _QMC_PRE_RICH_EVALUATE = evaluate
    _QMC_PRE_RICH_RENDER   = render_audio

# ---------- extra raga-style scales (all still built from the same chord/melody machinery) ----------
SCALES["Bhairav (raga, dawn)"] = dict(parent=[0,1,4,5,7,8,11], melody=[0,1,4,5,7,8,11],
                                      progs=[[0,1,0,3],[0,3,1,0],[0,1,3,0],[0,0,1,0]])
RAGA_SCALES = {"Bhairav": "Bhairav (raga, dawn)", "Yaman (Lydian)": "Lydian (dreamy)",
               "Bhairavi (Phrygian)": "Phrygian (dark)", "Kafi (Dorian)": "Dorian (jazzy minor)",
               "Asavari (natural minor)": "Natural minor (sad)", "Khamaj (Mixolydian)": "Mixolydian (bluesy major)"}

# ---------- user-facing settings (the panel at the bottom of this cell writes into RICH) ----------
RICH = dict(riff="Auto", arp="Auto", counter="Auto", orn="Auto", var="Auto", flavour="Auto", scale="Genre default")
LEVELS = {"Off": 0, "Light": 1, "Full": 2}
GENRE_RICH = {   # riff / arp / counter / orn(aments) / var(rhythmic variety): 0 off, 1 light, 2 full
  "Pop / cinematic (default)": dict(riff=1, arp=2, counter=2, orn=1, var=1, flavour="Western"),
  "Jazz (swing)":              dict(riff=0, arp=1, counter=2, orn=2, var=2, flavour="Western"),
  "EDM / folk-house":          dict(riff=2, arp=2, counter=1, orn=1, var=2, flavour="Western"),
  "Lo-fi hip-hop":             dict(riff=0, arp=2, counter=1, orn=1, var=1, flavour="Western"),
  "Ambient":                   dict(riff=0, arp=2, counter=1, orn=0, var=0, flavour="Western"),
  "Country":                   dict(riff=1, arp=2, counter=2, orn=1, var=1, flavour="Blues"),
  "Rock":                      dict(riff=2, arp=1, counter=1, orn=2, var=2, flavour="Blues"),
}
FLAVOURS = {"Western (turns, trills)": "Western", "Hindustani (murki / harkat / meend)": "Hindustani",
            "Blues-rock (bends, slides)": "Blues"}

def _rich_cfg(genre):
    g = dict(GENRE_RICH.get(genre, GENRE_RICH["Pop / cinematic (default)"]))
    for k in ("riff", "arp", "counter", "orn", "var"):
        if RICH[k] != "Auto": g[k] = LEVELS[RICH[k]]
    if RICH["flavour"] != "Auto": g["flavour"] = FLAVOURS.get(RICH["flavour"], RICH["flavour"])
    return g

# General-MIDI instrument per role and genre (zero-based programs).
PROG = {
 "Pop / cinematic (default)": dict(melody=0,  arp=46, counter=48, riff=27,  hook=0),
 "Jazz (swing)":              dict(melody=65, arp=11, counter=59, riff=26,  hook=65),
 "EDM / folk-house":          dict(melody=81, arp=80, counter=90, riff=81,  hook=81),
 "Lo-fi hip-hop":             dict(melody=4,  arp=4,  counter=73, riff=26,  hook=4),
 "Ambient":                   dict(melody=88, arp=46, counter=52, riff=88,  hook=88),
 "Country":                   dict(melody=40, arp=25, counter=25, riff=105, hook=40),
 "Rock":                      dict(melody=29, arp=27, counter=29, riff=30,  hook=29),
}

MIX_VOL = dict(melody=118, orn=118, hook=118, harmony=78, pad=68, bass=100, arp=78, guitar=82, riff=90, counter=84)
MIX_PAN = dict(melody=64, orn=64, hook=64, harmony=78, pad=64, bass=64, arp=42, guitar=84, riff=48, counter=90)

# ---------------- generic helpers ----------------
def _ladder(tonic, intervals, lo=24, hi=108):
    pcs = {(tonic + i) % 12 for i in intervals}
    return [m for m in range(lo, hi + 1) if m % 12 in pcs]

def _lstep(lad, pitch, k):
    """Move k scale steps from `pitch` on ladder `lad` (nearest note if pitch is not on the ladder)."""
    i = bisect.bisect_left(lad, pitch)
    if i >= len(lad) or (i > 0 and abs(lad[i - 1] - pitch) <= abs(lad[i] - pitch)): i = max(0, i - 1)
    return lad[int(np.clip(i + k, 0, len(lad) - 1))]

def q_bits(n):
    """n fresh Hadamard-QRNG bits (batched in circuits of <= 12 qubits)."""
    out = []
    while len(out) < n:
        m = min(12, n - len(out))
        out += [int(b) for b in qrng_bits(m)[::-1]]
    return out

def q_flags(n, p):
    """n independent yes/no decisions with P(yes) ~ p (3 QRNG bits each -> resolution 1/8)."""
    if n <= 0: return []
    b = q_bits(3 * n); thr = int(round(np.clip(p, 0, 1) * 8))
    return [(b[3*i] * 4 + b[3*i+1] * 2 + b[3*i+2]) < thr for i in range(n)]

def q_choose(weights):
    """Sample an index with the given weights: probabilities -> amplitudes (StatePreparation) -> measurement."""
    w = np.maximum(np.asarray(weights, float), 1e-9); n = len(w)
    nq = max(1, int(math.ceil(math.log2(max(2, n)))))
    pr = np.zeros(2 ** nq); pr[:n] = w / w.sum()
    qc = QuantumCircuit(nq, nq); qc.append(StatePreparation(np.sqrt(pr)), range(nq)); qc.measure(range(nq), range(nq))
    idx = int(run_circuit(qc)[0], 2)
    return idx if idx < n else int(np.argmax(pr[:n]))

# ---------------- quantum walk ----------------
def walk_circuit(steps, start=0, up_bias=0.5, nq=3):
    """Discrete-time quantum walk on a ring of 2**nq positions: coin qubit + 'shift' (controlled +1 / -1).
    The position is measured after every step (a monitored walk) -> a whole TRAJECTORY from one circuit."""
    c = nq
    qc = QuantumCircuit(nq + 1, nq * steps)
    for b in range(nq):
        if (start >> b) & 1: qc.x(b)
    if abs(up_bias - 0.5) > 1e-9: qc.ry(math.pi / 2 - 2 * math.asin(math.sqrt(up_bias)), c)   # pre-rotate so that AFTER the first Hadamard P(coin=1)=up_bias
    for t in range(steps):
        qc.h(c)                                                  # coin flip
        qc.mcx([c, 0, 1], 2); qc.ccx(c, 0, 1); qc.cx(c, 0)       # coin=1 : position + 1
        qc.x(c); qc.cx(c, 0); qc.ccx(c, 0, 1); qc.mcx([c, 0, 1], 2); qc.x(c)   # coin=0 : position - 1
        qc.measure(range(nq), range(nq * t, nq * (t + 1)))
    return qc

def quantum_walk_path(steps, start=0, up_bias=0.5, nq=3):
    qc = walk_circuit(steps, start, up_bias, nq)
    s = run_circuit(qc)[0][::-1]                                 # index = classical-bit number
    path = [sum(int(s[nq * t + b]) << b for b in range(nq)) for t in range(steps)]
    return path, qc

# ---------------- note-list utilities ----------------
def _bar_of(slot, slots): return int(slot // slots)

def _sounding(notes, slot):
    """pitch of the melody note sounding at `slot` (or the latest earlier one)."""
    best = None
    for s, d, m, v in notes:
        if s <= slot + 1e-6: best = m
        else: break
    return best

# ============================================================
# A. MOTIF RETURN  (the hook becomes a real repeated, diatonically-sequenced motif)
# ============================================================
def layer_motif(piece, cfg):
    """Turn the 'hook' into a REAL motif return: bar(s) 0.. are copied to bars 2, 6, 10 ... transposed diatonically
    (a melodic *sequence*) to follow the chord, and the plain melody is removed there (no more two melodies at once)."""
    genre = piece["genre"]
    if cfg["var"] == 0 or genre in ("Ambient", "Jazz (swing)"): return 0
    slots, bars, LADM = piece["slots"], piece["bars"], piece["_ladm"]
    L = 1 if sum(1 for n in piece["melody"] if n[0] < slots) >= 3 else 2          # sparse genres: a 2-bar motif
    src = {b: [n for n in piece["melody"] if b * slots <= n[0] < (b + 1) * slots] for b in range(L)}
    if sum(len(v) for v in src.values()) < 2: return 0
    mel, hook, moved = list(piece["melody"]), [], 0
    for start in range(2, bars - 1, 4):
        for b in range(L):
            bar = start + b
            if bar >= bars - 1: break
            shift = piece["degs"][bar] - piece["degs"][b]
            shift = shift - 7 if shift > 3 else (shift + 7 if shift < -3 else shift)
            mel = [n for n in mel if not (bar * slots <= n[0] < (bar + 1) * slots)]
            for s, d, m, v in src[b]:
                hook.append((s + (bar - b) * slots, d, _lstep(LADM, m, shift), max(40, int(v * 0.92))))
            moved += 1
    piece["melody"], piece["hook"] = sorted(mel), sorted(hook)
    return moved

# ============================================================
# B. FAST RHYTHMIC CHANGES  (16th-note runs, triplets, pushes, stop-time) chosen per bar by a quantum circuit
# ============================================================
MODE_NAMES = ["straight", "16th run", "triplet burst", "push / syncopation", "stop-time"]
def layer_rhythm(piece, cfg):
    lvl = cfg["var"]
    if lvl == 0: return []
    slots, bars, genre = piece["slots"], piece["bars"], piece["genre"]
    LADM = piece["_ladm"]; hook_bars = {_bar_of(h[0], slots) for h in piece.get("hook", [])} if piece["genre"] != "Ambient" else set()
    by_bar = {b: [] for b in range(bars)}
    for n in piece["melody"]: by_bar[_bar_of(n[0], slots)].append(n)
    modes = []
    for bar in range(bars):
        notes = sorted(by_bar[bar])
        if bar == bars - 1 or not notes: modes.append(0); continue
        base = [6, 2, .5, 1.5, .3] if lvl == 1 else [3, 3, 1.8, 2.5, .8]
        w = np.array(base, float)
        if bar % 4 == 3: w *= [0.5, 1.8, 1.3, 1.2, 2.2]            # phrase end: fills / stop-time
        if bar % 4 == 0: w *= [2.0, .8, .6, .8, .1]                # phrase start: keep it clear
        if genre in ("Rock", "EDM / folk-house"): w *= [.8, 1.4, .8, 1.2, 1.0]
        if genre in ("Lo-fi hip-hop", "Country"): w *= [1.2, .6, .9, 1.2, .8]
        if slots != 8: w[2] *= 1.2
        mode = q_choose(w); modes.append(mode)
        out = []
        if mode == 1:      # 16th-note pick-up run into the longest note
            cand = [n for n in notes if n[1] >= 2 and n[0] > bar * slots - 1]
            if cand:
                s, d, m, v = max(cand, key=lambda n: n[1]); down = (bar % 2 == 1)
                leads = [_lstep(LADM, m, (3 - i) if down else -(3 - i)) for i in range(3)]
                out = [(s, .5, leads[0], v - 8), (s + .5, .5, leads[1], v - 8), (s + 1, .5, leads[2], v - 8), (s + 1.5, d - 1.5, m, v)]
                notes = [n for n in notes if n != (s, d, m, v)] + out
        elif mode == 2:    # eighth-note TRIPLET burst (3 notes in 2 slots = 2/3 slot each)
            cand = [n for n in notes if n[1] >= 2]
            if cand:
                s, d, m, v = cand[0]; t = 2 / 3
                tri = [(s, t, m, v), (s + t, t, _lstep(LADM, m, 2), v - 10), (s + 2 * t, t, _lstep(LADM, m, 1), v - 10)]
                if d > 2: tri.append((s + 2, d - 2, m, v - 4))
                notes = [n for n in notes if n != (s, d, m, v)] + tri
        elif mode == 3:    # push: pull an on-beat note half a slot early (anticipation)
            notes = sorted(notes)
            for i in range(1, len(notes)):
                s, d, m, v = notes[i]; ps, pd, pm, pv = notes[i - 1]
                if (s - bar * slots) % 2 == 0 and s > bar * slots and pd >= 1 and abs(ps + pd - s) < 1e-6:
                    notes[i - 1] = (ps, pd - .5, pm, pv); notes[i] = (s - .5, d + .5, m, v + 4); break
        elif mode == 4:    # stop-time: the last two slots go silent (drums/bass punch the gap)
            cut = (bar + 1) * slots - 2
            kept = []
            for s, d, m, v in notes:
                if s >= cut: continue
                kept.append((s, min(d, cut - s), m, v))
            if kept: notes = kept
        by_bar[bar] = sorted(notes)
    piece["melody"] = sorted(n for b in range(bars) for n in by_bar[b])
    piece["stop_bars"] = [b for b, m in enumerate(modes) if m == 4]
    return modes

# ============================================================
# C. ORNAMENTS  (Western turns / Hindustani murki-harkat-khatka-meend / Blues bends)
# ============================================================
ORN_TYPES = {"Western": ["grace", "mordent", "turn", "trill"],
             "Hindustani": ["murki", "harkat", "khatka", "kan", "meend"],
             "Blues": ["bend", "slide", "hammer", "grace"]}
ORN_W = {"Western": [1.2, 1.4, 1.0, .6], "Hindustani": [2.0, 1.6, 1.0, 1.0, 1.4], "Blues": [1.6, 1.2, 1.3, .8]}
SEMI = 8192 / 12.0   # pitch-bend units per semitone when the bend range is +/-12

def layer_ornaments(piece, cfg):
    lvl = cfg["orn"]
    if lvl == 0: return {}
    slots, bars = piece["slots"], piece["bars"]
    flav = cfg["flavour"]; types, tw = ORN_TYPES[flav], ORN_W[flav]
    LAD = piece["_ladp"] if flav == "Hindustani" else piece["_ladm"]
    hook_bars = {_bar_of(h[0], slots) for h in piece.get("hook", [])}
    p_orn = (.20 if lvl == 1 else .45) * (1.25 if flav == "Hindustani" else 1.0)
    mel = sorted(piece["melody"]); new = []; orn, bends = [], []; counts = {}
    used_walk = False
    bar_idx = {}
    for i, n in enumerate(mel): bar_idx.setdefault(_bar_of(n[0], slots), []).append(i)
    skip = set()
    for bar in range(bars):
        ids = bar_idx.get(bar, [])
        elig = [i for i in ids if mel[i][1] >= 1 and bar < bars - 1 and i not in skip]
        flags = q_flags(len(elig), p_orn) if elig else []
        chosen = [i for i, f in zip(elig, flags) if f]
        for i in ids:
            if i in skip: continue
            s, d, m, v = mel[i]
            if i not in chosen:
                new.append(mel[i]); continue
            typ = types[q_choose(tw)]
            u = lambda k: _lstep(LAD, m, k)
            sign = 1 if (bar + i) % 2 == 0 else -1
            done = None
            if typ in ("murki", "harkat", "khatka"):
                path, qcw = quantum_walk_path(4, start=4, up_bias=.5)
                if not used_walk: piece["rich_circuits"].append(("Quantum walk -> murki / harkat contour (4 steps, position measured each step)", qcw)); used_walk = True
                offs = [int(np.clip(p - 4, -2, 2)) for p in path]
                fast = []
                for o in offs:
                    nxt = u(o)
                    if (fast and nxt == fast[-1]) or (not fast and nxt == m): continue
                    fast.append(nxt)
                if typ == "murki" and d >= 1 and fast:
                    nf = min(3, len(fast), int((d - .75) / .25))
                    if nf >= 1:
                        fast = fast[:nf]; used = .5 + .25 * nf
                        done = [(s, .5, m, v)] + [(s + .5 + .25 * j, .25, p, v - 12) for j, p in enumerate(fast)] + [(s + used, d - used, m, v - 4)]
                elif typ == "harkat" and d >= 1 and len(fast) >= 3:
                    f3 = fast[:3]; st = .5 if d >= 2 else .25          # a harkat is a quick twisting run into the note
                    done = [(s + st * j, st, p, v - 8) for j, p in enumerate(f3)] + [(s + 3 * st, d - 3 * st, m, v)]
                elif typ == "khatka" and d >= 1.5:
                    up = u(1)
                    done = [(s, .25, up, v - 10), (s + .25, .25, m, v - 6), (s + .5, .25, up, v - 10), (s + .75, .25, m, v - 6), (s + 1, d - 1, m, v - 2)]
            elif typ in ("kan", "grace", "hammer"):
                g = u(sign if typ != "hammer" else -1)
                done = [(s, .25, g, v - 14), (s + .25, d - .25, m, v)]
            elif typ == "mordent" and d >= 1:
                done = [(s, .25, m, v), (s + .25, .25, u(1), v - 10), (s + .5, d - .5, m, v - 4)]
            elif typ == "turn" and d >= 1:
                done = [(s, .25, u(1), v - 10), (s + .25, .25, m, v - 6), (s + .5, .25, u(-1), v - 10), (s + .75, d - .75, m, v - 2)]
            elif typ == "trill" and d >= 2:
                k = int((min(d, 2) - .5) / .25)
                done = [(s + .25 * j, .25, m if j % 2 == 0 else u(1), v - 6) for j in range(k)] + [(s + .25 * k, d - .25 * k, m, v - 2)]
            elif typ == "meend" and i + 1 in range(len(mel)) and (i + 1) not in skip:
                s2, d2, m2, v2 = mel[i + 1]
                if abs(s + d - s2) < 1e-6 and 0 < abs(m2 - m) <= 5 and d >= 1 and _bar_of(s2, slots) == bar:
                    orn.append((s, d + d2, m, v)); skip.add(i + 1); t0 = s + .35 * d; steps = 6
                    for j in range(steps + 1):
                        bends.append((t0 + (s + d - t0) * j / steps, int(np.clip((m2 - m) * SEMI * j / steps, -8191, 8191))))
                    bends.append((s + d + d2, 0)); counts["meend"] = counts.get("meend", 0) + 1; continue
            elif typ in ("bend", "slide") and d >= 1:
                dep = 2 if typ == "bend" else 3; span = .5 if typ == "bend" else .25
                orn.append((s, d, m, v))
                bends.append((s - 0.01, int(-dep * SEMI)))
                for j in range(1, 5): bends.append((s + span * j / 4, int(-dep * SEMI * (1 - j / 4))))
                counts[typ] = counts.get(typ, 0) + 1; continue
            if done:
                done = [x for x in done if x[1] > 1e-6]
                new.extend(done); counts[typ] = counts.get(typ, 0) + 1
            else:
                new.append(mel[i])
    piece["melody"] = sorted(new); piece["orn"] = sorted(orn); piece["bends"] = sorted(bends)
    return counts

# ============================================================
# D. QUANTUM-WALK ARPEGGIOS
# ============================================================
ARP_GRID = {"Pop / cinematic (default)": (.5, 1), "EDM / folk-house": (.5, .5), "Jazz (swing)": (1, 1),
            "Lo-fi hip-hop": (1, 1), "Ambient": (1, 2), "Country": (1, 1), "Rock": (1, 1)}
ARP_DENS = {"Lo-fi hip-hop": .70, "Ambient": .55, "Jazz (swing)": .75, "Rock": .85}

def _arp_active(genre, bar, bars, lvl):
    if genre == "EDM / folk-house": return bar >= int(bars * .20)
    if genre == "Ambient": return bar % 2 == 0
    if genre == "Rock": return bar % 8 < 4 or bar >= bars - 2          # clean arpeggio under the verse, riff takes over later
    if lvl == 1: return bar % 4 != 3
    return True

def layer_arp(piece, cfg):
    lvl = cfg["arp"]
    if lvl == 0: return 0
    slots, bars, genre, CH = piece["slots"], piece["bars"], piece["genre"], piece["CH"]
    grid = ARP_GRID.get(genre, (1, 1))[0 if lvl == 2 else 1]
    dens = ARP_DENS.get(genre, 1.0)
    arp, pos, shown = [], 2, False
    for bar in range(bars):
        if not _arp_active(genre, bar, bars, lvl) and bar != bars - 1: continue
        tones = CH[piece["degs"][bar]]["tones"]
        lad5 = [m for m in range(52, 96) if m % 12 in set(tones)][:5]       # 5 chord pitches going up
        steps = int(round(slots / grid))
        path, qcw = quantum_walk_path(steps, start=pos, up_bias=.7 if bar % 2 == 0 else .35)
        if not shown: piece["rich_circuits"].append((f"Quantum-walk arpeggio, bar 1 ({steps} steps over a 5-note chord ladder)", qcw)); shown = True
        pos = path[-1]
        keep = q_flags(steps, dens) if dens < 1 else [True] * steps
        for k, p in enumerate(path):
            if not keep[k] and k % int(max(1, round(2 / grid))) != 0: continue
            idx = p if p <= 4 else 8 - p                                    # fold the ring into a zig-zag ladder
            on_beat = (k * grid) % 2 == 0
            arp.append((bar * slots + k * grid, grid * .92, lad5[min(idx, len(lad5) - 1)], 70 if on_beat else 48))
    piece["arp"] = arp
    return len(arp)

# ============================================================
# E. COUNTER-MELODY  (Grover consonance oracle + Bell-pair entangled echo)
# ============================================================
def _counter_active(genre, bar, bars, lvl):
    if bar == bars - 1: return False
    base = {"Pop / cinematic (default)": bar % 8 >= 4 or bar >= bars - 4, "Jazz (swing)": bar % 2 == 0,
            "EDM / folk-house": bar >= int(bars * .45), "Lo-fi hip-hop": bar % 4 >= 2, "Ambient": bar % 4 >= 2,
            "Country": bar % 2 == 1, "Rock": bar % 8 >= 4}.get(genre, True)
    return base and (lvl == 2 or bar % 2 == 1)

def layer_counter(piece, cfg):
    lvl = cfg["counter"]
    if lvl == 0: return 0
    slots, bars, genre, CH, ts = piece["slots"], piece["bars"], piece["genre"], piece["CH"], piece["ts"]
    LADM = piece["_ladm"]
    full = sorted(piece["melody"] + piece.get("hook", []) + piece.get("orn", []))
    cands = [m for m in LADM if 50 <= m <= 80]; mid = cands.index(min(cands, key=lambda x: abs(x - 62)))
    n = 10 if len(cands) >= 10 else len(cands)
    lo = int(np.clip(mid - n // 2, 0, len(cands) - n)); CM = cands[lo:lo + n]
    nq = max(2, int(math.ceil(math.log2(len(CM)))))
    out, prev_i, prev_M, shown, n_echo, n_free = [], len(CM) // 2, None, False, 0, 0
    for bar in range(bars):
        if not _counter_active(genre, bar, bars, lvl): continue
        t0 = bar * slots; tones = set(CH[piece["degs"][bar]]["tones"])
        bar_mel = [n for n in full if t0 <= n[0] < t0 + slots]
        if not bar_mel: continue
        echo = bool(piece["bell"][bar][0])      # Bell pair (q0 == q1): one bit shared between melody and counter-melody
        if echo:
            for s, d, m, v in bar_mel:
                if s + 2 < t0 + slots and d >= .5:
                    out.append((s + 2, min(max(d, 1), 2), _lstep(LADM, m, -2), 54)); n_echo += 1
            continue
        onsets = {round(n[0] - t0) for n in bar_mel}
        anchors = [0] + ([slots // 2] if slots >= 6 else [])
        free = [s for s in range(slots) if s not in onsets and s not in anchors]
        pick = [s for s, f in zip(free, q_flags(len(free), .35)) if f]
        sl = sorted(set(anchors) | set(pick))
        for j, s in enumerate(sl):
            M = _sounding(full, t0 + s)
            if M is None: continue
            strong = s in ts["strong"]
            w = np.zeros(2 ** nq); marked = []
            for i, c in enumerate(CM):
                dist = abs(i - prev_i); x = {0: .3, 1: 2.6, 2: 1.8, 3: .7}.get(dist, .25)
                iv = (M - c) % 12
                if iv in (3, 4, 8, 9): x *= 1.6
                if iv == 0: x *= .1
                if prev_M is not None and np.sign(c - CM[prev_i]) == -np.sign(M - prev_M) and dist: x *= 1.7   # contrary motion
                x *= math.exp(-((i - len(CM) / 2) ** 2) / (2 * (len(CM) * .4) ** 2))
                w[i] = x
                if iv in (3, 4, 8, 9) and c % 12 in tones: marked.append(i)
            if not marked: marked = [i for i, c in enumerate(CM) if (M - c) % 12 in (3, 4, 8, 9)]
            pr = w / w.sum()
            adj = strong_prior(pr, marked) if (strong and 0 < len(marked) < len(CM)) else None
            if adj is not None:
                qc, k, pb = amplitude_amplification_circuit(adj, marked); used = adj
            else:
                qc, k, pb = amplitude_amplification_circuit(pr, marked, k=0); used = pr
            idx = int(run_circuit(qc)[0], 2)
            if idx >= len(CM): idx = int(np.argmax(used[:len(CM)]))
            if k > 0 and not shown:
                piece["rich_circuits"].append((f"Counter-melody: Grover consonance oracle (k={k}) boosts 3rds/6ths against the lead", qc)); shown = True
            nxt = sl[j + 1] if j + 1 < len(sl) else slots
            out.append((t0 + s, min(max(nxt - s, 1), 4), CM[idx], 58 if strong else 50))
            prev_i, prev_M = idx, M; n_free += 1
    piece["counter"] = out; piece["counter_stats"] = (n_free, n_echo)
    return len(out)

# ============================================================
# F. RIFFS  (PQCA proposes 8 riffs -> Grover picks one of the 2 best -> quantum-sampled pitches)
# ============================================================
RIFF_DENS = {"Rock": (7, 11), "EDM / folk-house": (8, 12), "Pop / cinematic (default)": (4, 7), "Country": (5, 8)}
RIFF_ALPHA = {   # (semitone offset | 'third' | 'fifth' | 'b7', weight)
  "Rock":    [(0, 7), ("third", 1.3), (5, .9), ("fifth", 2.4), (10, 1.6), (12, 1.2), (6, .35)],
  "EDM / folk-house": [(0, 3), ("third", 2), ("fifth", 2.2), (12, 2), (2, .6)],
  "Pop / cinematic (default)": [(0, 3), ("third", 2), ("fifth", 2.5), (12, 1), (2, .7)],
  "Country": [(0, 3), ("third", 2), ("fifth", 2.5), (9, 1), (12, 1)],
}
def _riff_active(genre, bar, bars, lvl):
    if bar == bars - 1: return False
    if genre == "Rock": return bar >= 1 and (bar % 8 >= 4 or bar >= 2) and not (lvl == 1 and bar % 2)
    if genre == "EDM / folk-house": return bar >= int(bars * .55)
    return bar % 2 == 0 if lvl == 1 else bar % 4 != 3

def layer_riff(piece, cfg):
    lvl = cfg["riff"]; genre = piece["genre"]
    if lvl == 0 or genre not in RIFF_DENS: return None
    slots, bars, CH = piece["slots"], piece["bars"], piece["CH"]
    n16 = slots * 2; lo, hi = RIFF_DENS[genre]; lo = max(2, round(lo * n16 / 16)); hi = max(lo + 1, round(hi * n16 / 16))
    # 1. PQCA proposes 8 candidate riff rhythms on a 16th-note grid
    state = [1 if i in (0, 3, 6, 10) else 0 for i in range(n16)]; cands = []
    for _ in range(8):
        for _t in range(15):
            nxt = pqca_step(state)
            if lo <= sum(nxt) <= hi: break
        state = nxt; pat = nxt.copy(); pat[0] = 1; cands.append(pat)
    def score(p):
        on = [i for i, b in enumerate(p) if b]; gaps = np.diff(on + [n16]) if on else [n16]
        sync = sum(1 for i in on if i % 2) + .5 * sum(1 for i in on if i % 4 == 2)
        return sync - .6 * max(0, max(gaps) - 6) + (1 if lo <= len(on) <= hi else -3)
    order = sorted(range(8), key=lambda i: -score(cands[i])); marked = sorted(order[:2])
    # 2. Grover search: 3 qubits, 2 marked of 8 -> one iteration lifts P(marked) from 25 % to ~100 %
    qc, k, p = amplitude_amplification_circuit(np.ones(8) / 8, marked)
    pick = int(run_circuit(qc)[0], 2); pick = pick if pick < 8 else marked[0]
    piece["rich_circuits"].append((f"Riff search: Grover (k={k}) finds one of the 2 best PQCA riffs; candidate #{pick} chosen", qc))
    pat = cands[pick]; on = [i for i, b in enumerate(pat) if b]
    # 3. riff pitches: sampled ONCE as offsets relative to the chord root (so the riff transposes with the chords)
    alpha = RIFF_ALPHA[genre]
    if genre == "Rock" and cfg["flavour"] != "Blues": alpha = [a for a in alpha if a[0] != 6]
    def draw(i):
        on_beat = on[i] % 4 == 0
        ws = [w * (1 if (not on_beat or a in (0, "third", "fifth", 12)) else 0) for a, w in alpha]
        if not any(ws): ws[0] = 1
        return q_choose(ws)
    motif = [draw(i) for i in range(len(on))]
    alt = list(motif)
    for i in range(max(0, len(on) - 2), len(on)): alt[i] = draw(i)      # turnaround variation (every 4th bar)
    riff, bass_new, guit = [], [], []
    for bar in range(bars):
        if not _riff_active(genre, bar, bars, lvl): continue
        ch = CH[piece["degs"][bar]]; t0 = bar * slots; r = ch["root"]
        third = (ch["tones"][1] - r) % 12; fifth = (ch["tones"][2] - r) % 12
        base = {"Rock": 40, "EDM / folk-house": 60, "Pop / cinematic (default)": 55, "Country": 55}[genre]
        root = base + ((r - base) % 12)
        shape = alt if bar % 4 == 3 else motif
        for j, i16 in enumerate(on):
            a = alpha[shape[j]][0]; off = third if a == "third" else (fifth if a == "fifth" else a)
            nxt = on[j + 1] if j + 1 < len(on) else n16
            dur = max(.25, (nxt - i16) * .5 * (.8 if genre == "Rock" else .9)); accent = i16 % 4 == 0
            vel = (104 if accent else 74) if genre == "Rock" else (86 if accent else 62)
            pitch = root + off
            riff.append((t0 + i16 * .5, dur, pitch, vel))
            if genre == "Rock" and accent: riff.append((t0 + i16 * .5, dur, pitch + 7, vel - 8))   # power-chord fifth
            if genre == "Rock" and off == 0: bass_new.append((t0 + i16 * .5, max(.5, dur), 28 + ((r - 28) % 12), 92 if accent else 70))
        if genre == "Rock":
            guit.append((t0, slots, root + 12, 50)); guit.append((t0, slots, root + 19, 50))
    piece["riff"] = riff; piece["riff_info"] = dict(k=k, pick=pick, onsets=len(on), marked=marked)
    if genre == "Rock":
        riff_bars = {_bar_of(n[0], slots) for n in riff}
        piece["bass"] = [b for b in piece["bass"] if _bar_of(b[0], slots) not in riff_bars] + bass_new
        piece["guitar"] = [g for g in piece.get("guitar", []) if _bar_of(g[0], slots) not in riff_bars] + guit
    return pick

# ============================================================
# G. DRUMS  (fills, crashes, 16th hats, ghost notes, double-kick on riff hits)
# ============================================================
def layer_drums(piece, cfg):
    lvl = cfg["var"]; genre = piece["genre"]
    if lvl == 0 or genre == "Ambient" or not piece.get("drums"): return 0
    slots, bars = piece["slots"], piece["bars"]; dr = list(piece["drums"]); added = 0
    soft = genre in ("Lo-fi hip-hop", "Jazz (swing)")
    cap = (lambda v: min(int(v), 60)) if genre == "Lo-fi hip-hop" else (lambda v: int(min(v, 127)))
    stop = set(piece.get("stop_bars", []))
    for bar in range(bars - 1):
        t0 = bar * slots; end = t0 + slots
        if bar % 4 == 3:                                                  # fill at the end of every 4-bar phrase
            kind = q_choose([3, 2, 2, 2, 1] if lvl == 2 else [3, 1.5, 1, 1, 1.5]); f0 = end - 2
            if bar in stop: kind = 4
            dr = [e for e in dr if not (f0 - 1e-6 <= e[0] < end and e[2] != 51)]
            ev = []
            if kind == 0: ev = [(f0 + .5 * j, .5, 38, 48 + 18 * j) for j in range(4)]                      # snare roll (16ths, crescendo)
            elif kind == 1: ev = [(f0 + .5 * j, .5, p, 92 + 4 * j) for j, p in enumerate([50, 48, 47, 45])] # tom run down
            elif kind == 2: ev = [(f0, .5, 36, 100), (f0 + .5, .5, 38, 80), (f0 + 1, .5, 36, 96), (f0 + 1.5, .5, 38, 112)]  # kick/snare syncopation
            elif kind == 3: ev = [(f0 + j / 3, 1 / 3, 38, 50 + 12 * j) for j in range(6)]                    # triplet roll
            elif kind == 4: ev = [(f0 + 1.5, .5, 38, 70)]                                                    # stop-time pick-up
            if soft: ev = [(s, d, p, min(v, 58)) for s, d, p, v in ev]
            dr += [(s, d, p, cap(v)) for s, d, p, v in ev]; added += len(ev)
        if bar % 4 == 0 and bar > 0 and genre in ("Rock", "EDM / folk-house", "Pop / cinematic (default)", "Country"):
            dr.append((t0, 1, 49, 100)); added += 1                      # crash on the phrase downbeat
        if lvl == 2 and genre in ("Rock", "EDM / folk-house", "Pop / cinematic (default)") and bar % 4 != 3:
            for s in range(slots):                                         # 16th hats between the 8ths
                dr.append((t0 + s + .5, .5, 42, 20 + 6 * (s % 2))); added += 1
        if genre in ("Lo-fi hip-hop", "Jazz (swing)", "Country") and bar % 4 != 3:
            fl = q_flags(slots, .22 if genre != "Country" else .12)
            for s, f in zip(range(slots), fl):
                if f: dr.append((t0 + s + .5, .5, 38 if genre != "Lo-fi hip-hop" else 37, 24)); added += 1   # ghost notes
    if genre == "Rock" and piece.get("riff"):                               # double kick: lock kicks to off-beat riff hits
        have = {round(e[0], 3) for e in dr if e[2] == 36}
        cand = [r for r in piece["riff"] if r[3] < 90 and abs(r[0] * 2 - round(r[0] * 2)) < 1e-6 and int(round(r[0] * 2)) % 2 == 1
                and round(r[0], 3) not in have and _bar_of(r[0], slots) % 4 in (2, 3) and _bar_of(r[0], slots) < bars - 1]
        for r, f in zip(cand, q_flags(len(cand), .5)):
            if f: dr.append((r[0], .5, 36, 88)); added += 1
    piece["drums"] = sorted(dr, key=lambda e: e[0])
    return added

# ============================================================
# MAIN: enrich the skeleton
# ============================================================
def compose(p, progress=lambda frac, msg="": None):
    p = dict(p)
    if RICH.get("scale", "Genre default") in RAGA_SCALES: p["scale"] = RAGA_SCALES[RICH["scale"]]
    piece = _QMC_PRE_RICH_COMPOSE(p, progress)
    cfg = _rich_cfg(piece["genre"]); piece["rich_cfg"] = cfg
    sc, tonic = piece["sc"], piece["tonic"]
    piece["_ladm"] = _ladder(tonic, sc["melody"]); piece["_ladp"] = _ladder(tonic, sc["parent"])
    piece["melody_core"] = list(piece["melody"]); piece["rich_circuits"] = []
    piece.update(riff=[], counter=[], orn=[], bends=[], stop_bars=[])
    stats = {}
    progress(.93, "Quantum walk builds arpeggios ...");           stats["arp"] = layer_arp(piece, cfg)
    progress(.94, "Motif return + rhythmic variation ...");       stats["motif_bars"] = layer_motif(piece, cfg); stats["modes"] = layer_rhythm(piece, cfg)
    progress(.95, "Ornaments: murki / harkat / meend ...");       stats["orn"] = layer_ornaments(piece, cfg)
    progress(.96, "Grover builds the counter-melody ...");        stats["counter"] = layer_counter(piece, cfg)
    progress(.97, "Grover searches riffs ...");                   stats["riff"] = layer_riff(piece, cfg)
    progress(.98, "Drum fills ...");                              stats["drums"] = layer_drums(piece, cfg)
    piece["rich_stats"] = stats
    return piece

# ============================================================
# MIDI export with all the new tracks, pitch-bend (meend / bends) and fractional-slot timing
# ============================================================
def midi_bytes(piece, filename="quantum_music.mid"):
    mid = MidiFile(type=1, ticks_per_beat=480)
    bpm, ts, genre, swing = piece["p"]["bpm"], piece["ts"], piece["genre"], float(piece.get("style", {}).get("swing", .5))
    P = PROG.get(genre, PROG["Pop / cinematic (default)"])
    pad_prog = 0 if genre == "Jazz (swing)" else (4 if genre == "Lo-fi hip-hop" else 89)
    bass_prog = 32 if genre == "Jazz (swing)" else (38 if genre == "EDM / folk-house" else 33)
    gtr_prog = 65 if genre == "Jazz (swing)" else (24 if genre == "Pop / cinematic (default)" else (29 if genre == "Rock" else 25))
    tracks = [("melody", piece["melody"], 0, P["melody"]), ("harmony", piece["harmony"], 4, 0 if genre != "Rock" else 29),
              ("pad", piece["pad"], 1, pad_prog), ("bass", piece["bass"], 2, bass_prog), ("arp", piece["arp"], 3, P["arp"]),
              ("guitar", piece.get("guitar", []), 5, gtr_prog), ("hook", piece.get("hook", []), 6, P["hook"]),
              ("riff", piece.get("riff", []), 7, P["riff"]), ("counter", piece.get("counter", []), 8, P["counter"]),
              ("orn", piece.get("orn", []), 10, P["melody"]), ("drums", piece.get("drums", []), 9, None)]
    swung = {"Jazz (swing)": {"melody", "guitar", "counter", "orn", "arp", "hook"},
             "Lo-fi hip-hop": {"melody", "arp", "hook", "counter", "orn"}}.get(genre, set())
    def tick(slot, name, pitch):
        base = int(round(slot * 240))
        if swing > .5 and abs(slot - round(slot)) < 1e-6 and int(round(slot)) % 2 == 1:
            if name in swung or (name == "drums" and pitch in (51, 42) and genre in ("Jazz (swing)", "Lo-fi hip-hop")):
                return base + int(round((swing - .5) * 480))
        return base
    for name, notes, ch, prog in tracks:
        tr = MidiTrack(); mid.tracks.append(tr); tr.append(MetaMessage("track_name", name=name, time=0))
        if name == "melody":
            tr.append(MetaMessage("set_tempo", tempo=bpm2tempo(bpm), time=0))
            tr.append(MetaMessage("time_signature", numerator=ts["num"], denominator=ts["den"], time=0))
        if prog is not None: tr.append(Message("program_change", channel=ch, program=int(prog), time=0))
        if name != "drums":                                            # simple mix: level + stereo position per role
            tr.append(Message("control_change", channel=ch, control=7, value=MIX_VOL.get(name, 90), time=0))
            tr.append(Message("control_change", channel=ch, control=10, value=MIX_PAN.get(name, 64), time=0))
        if name == "orn":      # pitch-bend range = +/-12 semitones (RPN 0,0) so meends / bends are accurate
            for c, v in ((101, 0), (100, 0), (6, 12), (38, 0), (101, 127), (100, 127)):
                tr.append(Message("control_change", channel=ch, control=c, value=v, time=0))
        ev = []
        for s, d, pitch, vel in notes:
            st = tick(s, name, pitch); en = max(st + 1, tick(s + d, name, pitch) - 5)
            ev.append((st, 2, Message("note_on", channel=ch, note=int(pitch), velocity=int(np.clip(vel, 1, 127)))))
            ev.append((en, 0, Message("note_off", channel=ch, note=int(pitch), velocity=0)))
        if name == "orn":
            for s, val in piece.get("bends", []):
                ev.append((max(0, int(round(s * 240))), 1, Message("pitchwheel", channel=ch, pitch=int(np.clip(val, -8192, 8191)))))
        if genre == "EDM / folk-house" and name == "pad" and piece.get("S", {}).get("sidechain", True):
            for s, d, pitch, vel in piece.get("drums", []):
                if pitch == 36:
                    t = tick(s, "drums", pitch)
                    ev.append((t, 1, Message("control_change", channel=ch, control=7, value=58)))
                    ev.append((t + 75, 1, Message("control_change", channel=ch, control=7, value=100)))
        ev.sort(key=lambda e: (e[0], e[1])); last = 0
        for t, _, msg in ev:
            msg.time = max(0, t - last); tr.append(msg); last = t
    mid.save(filename)
    with open(filename, "rb") as f: return f.read()

# fractional slots in the FM fallback renderer + merging the new tracks into the old track names
def _slot_seconds(slot, slot_sec, swing):
    whole = int(math.floor(slot + 1e-9)); frac = slot - whole
    sw = (2.0 * (swing - 0.5) * slot_sec) if (swing > 0.5 and whole % 2 == 1 and abs(frac) < 1e-6) else 0.0
    return whole * slot_sec + frac * slot_sec + sw

def render_audio(piece, renderer="soundfont"):
    if renderer == "soundfont" and SOUNDFONT_PATH and FLUIDSYNTH_PATH:
        return _QMC_PRE_RICH_RENDER(piece, renderer)
    p2 = dict(piece)                                                       # FM fallback: fold new tracks into old ones
    p2["melody"] = sorted(list(piece["melody"]) + list(piece.get("hook", [])) + list(piece.get("orn", [])))
    p2["harmony"] = list(piece["harmony"]) + list(piece.get("counter", []))
    p2["guitar"] = list(piece.get("guitar", [])) + list(piece.get("riff", []))
    return _QMC_PRE_RICH_RENDER(p2, "fm")

# ============================================================
# Evaluation: the old checks run on the ORIGINAL quantum melody; new layers get their own checks
# ============================================================
def evaluate(piece, audio=None):
    core = dict(piece); core["melody"] = piece.get("melody_core", piece["melody"])
    ev = _QMC_PRE_RICH_EVALUATE(core, audio)
    cfg, st = piece.get("rich_cfg", {}), piece.get("rich_stats", {})
    mel = sorted(piece["melody"] + piece.get("hook", []))
    durs = sorted({round(n[1], 2) for n in mel}); n_fast = sum(1 for n in mel if n[1] <= .5 + 1e-6)
    on_per_sec = len(mel) / max(1e-9, piece["total"])
    chk = ev["genre_checks"]
    chk["Fast rhythmic values (16th / triplet / 32nd) present"] = n_fast >= max(4, piece["bars"] // 2) if cfg.get("var", 0) or cfg.get("orn", 0) else True
    chk["Melody has >= 4 distinct note lengths"] = len(durs) >= 4
    if cfg.get("arp"):     chk["Quantum-walk arpeggio layer"] = len(piece["arp"]) >= piece["bars"]
    if cfg.get("counter"): chk["Counter-melody layer"] = len(piece.get("counter", [])) > 0
    if cfg.get("riff") and piece["genre"] in RIFF_DENS: chk["Riff layer (Grover-selected)"] = len(piece.get("riff", [])) > 0
    if cfg.get("orn"):     chk["Ornaments (murki / turns / bends)"] = sum(st.get("orn", {}).values()) > 0
    ev["genre_score"] = 100 * sum(bool(v) for v in chk.values()) / max(1, len(chk))
    ml = ev.get("melody_metrics", {}); ml["notes_per_second"] = on_per_sec; ev["melody_metrics"] = ml
    modes = st.get("modes") or []
    mcount = {MODE_NAMES[i]: modes.count(i) for i in range(len(MODE_NAMES)) if modes.count(i)}
    rinfo = piece.get("riff_info")
    ev["melody_strategy"] = (ev.get("melody_strategy", "") +
        f" | RICH LAYERS -> arpeggio notes: {len(piece['arp'])}, counter-melody notes: {len(piece.get('counter', []))} "
        f"(free/Grover {piece.get('counter_stats', (0,0))[0]}, Bell-echo {piece.get('counter_stats', (0,0))[1]}), "
        f"riff notes: {len(piece.get('riff', []))}" + (f" (Grover k={rinfo['k']}, PQCA candidate #{rinfo['pick']})" if rinfo else "") +
        f", ornaments: {st.get('orn', {})}, bar rhythm modes: {mcount}, melody notes/sec: {on_per_sec:.1f}")
    return ev

# fuller piano roll
def fig_piano_roll(piece):
    fig, ax = plt.subplots(figsize=(13, 5)); ss = piece["slot_sec"]; slots = piece["slots"]
    layers = [(piece["pad"], "#c7e9c0"), (piece["bass"], "#1f77b4"), (piece["arp"], "#9ecae1"), (piece.get("guitar", []), "#bcbd22"),
              (piece.get("riff", []), "#8c564b"), (piece.get("counter", []), "#2ca02c"), (piece["harmony"], "#ff9896"),
              (piece.get("hook", []), "#e377c2"), (piece["melody"], "#d62728"), (piece.get("orn", []), "#ff7f0e")]
    for notes, col in layers:
        for s, d, pitch, v in notes: ax.add_patch(Rectangle((s * ss, pitch - .4), d * ss * .95, .8, color=col))
    for b in range(piece["bars"] + 1): ax.axvline(b * slots * ss, color="gray", lw=.4)
    for b, c in enumerate(piece["labels"]): ax.text(b * slots * ss + .05, 24.5, c, fontsize=8)
    ax.set_xlim(0, piece["total"]); ax.set_ylim(24, 100); ax.set_xlabel("seconds"); ax.set_ylabel("MIDI pitch")
    ax.set_title("red: melody · orange: meend/bend · pink: motif return · green: counter-melody · brown: riff · light-blue: quantum-walk arpeggio · blue: bass")
    plt.tight_layout(); return fig

# ============================================================
# Control panel for the new layers (set values, then press Compose in the dashboard above)
# ============================================================
LAST = {}
_orig_compose_for_last = compose
def compose(p, progress=lambda frac, msg="": None):       # remember the latest piece so the circuit viewer can draw it
    piece = _orig_compose_for_last(p, progress); LAST["piece"] = piece; return piece

def make_rich_panel():
    opt = ["Auto", "Off", "Light", "Full"]; sty = dict(description_width="165px"); lay = W.Layout(width="330px")
    wd = {k: W.Dropdown(options=opt, value="Auto", description=d, style=sty, layout=lay) for k, d in
          [("riff", "Riffs"), ("arp", "Quantum-walk arpeggios"), ("counter", "Counter-melody"),
           ("orn", "Ornaments"), ("var", "Rhythmic variety / fills")]}
    wf = W.Dropdown(options=["Auto"] + list(FLAVOURS), value="Auto", description="Ornament flavour", style=sty, layout=lay)
    ws = W.Dropdown(options=["Genre default"] + list(RAGA_SCALES), value="Genre default", description="Raga / scale override", style=sty, layout=lay)
    out = W.Output(); btn = W.Button(description="Show new quantum circuits of the last piece", layout=W.Layout(width="360px"))
    def sync(_=None):
        for k, w in wd.items(): RICH[k] = w.value
        RICH["flavour"] = wf.value; RICH["scale"] = ws.value
    for w in list(wd.values()) + [wf, ws]: w.observe(sync, names="value")
    def show(_=None):
        out.clear_output()
        with out:
            pc = LAST.get("piece")
            if not pc: print("Compose a piece first."); return
            for cap, qc in pc.get("rich_circuits", []):
                display(HTML(f"<h4>{cap}</h4>"))
                try: fig = qc.draw("mpl", fold=-1); display(fig); plt.close(fig)
                except Exception: print(qc.draw("text"))
    btn.on_click(show)
    return W.VBox([W.HTML("<h3>🎛️ Rich-phrase controls</h3><p>Set these, then press <b>🎼 Compose</b> in the dashboard above. "
                          "<i>Auto</i> = sensible per-genre default. Try <b>Ornament flavour → Hindustani</b> with <b>Raga → Bhairav</b>.</p>"),
                   W.HBox([W.VBox(list(wd.values())[:3]), W.VBox(list(wd.values())[3:] + [wf, ws])], layout=W.Layout(flex_flow="row wrap")), btn, out])

display(make_rich_panel())
print("Rich phrase engine loaded: riffs, quantum-walk arpeggios, counter-melody, fast rhythm, murki/harkat/meend.")


Rich phrase engine loaded: riffs, quantum-walk arpeggios, counter-melody, fast rhythm, murki/harkat/meend.
